<a href="https://colab.research.google.com/github/penda54/terrafisc1/blob/main/TERRAFISC_v4_(3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏛️ TERRAFISC — Système de Gestion Administrative
**Projet SIDBR 2025-2026 | Prof. Papa DIOP**

### 5 Bureaux : GCS · Recouvrement · Domaines · Conservation Foncière · Cadastre
### Fonctionnalités : Hiérarchie N+1 · Employé du mois · Télépaiement · Notifications sonores

**Exécuter dans l'ordre : Cellule 1 → 2 → 3 → 4**

## 📦 Cellule 1 — Installation des dépendances

In [1]:
!pip install flask flask-cors pyngrok --quiet
print('✅ Dépendances installées')

✅ Dépendances installées


## 🗄️ Cellule 2 — Base de données & données initiales

In [3]:
import sqlite3, hashlib
from datetime import date
DB = 'terrafisc.db'

def get_db():
    c = sqlite3.connect(DB)
    c.row_factory = sqlite3.Row
    c.execute('PRAGMA foreign_keys=ON')
    return c

def h(p): return hashlib.sha256(p.encode()).hexdigest()

def init_db():
    conn = get_db(); c = conn.cursor()
    c.executescript('''
    CREATE TABLE IF NOT EXISTS bureau(
        bureau_id    INTEGER PRIMARY KEY AUTOINCREMENT,
        nom          VARCHAR(80) NOT NULL,
        code         VARCHAR(10),
        description  TEXT
    );
    CREATE TABLE IF NOT EXISTS personnel(
        personnel_id    INTEGER PRIMARY KEY AUTOINCREMENT,
        nom             VARCHAR(30) NOT NULL,
        prenom          VARCHAR(30),
        email           TEXT UNIQUE NOT NULL,
        mot_de_passe    TEXT NOT NULL,
        telephone       VARCHAR(15),
        dateNaissance   DATE,
        fonction        TEXT,
        role            TEXT NOT NULL CHECK(role IN ("chef_centre","chef_bureau","controleur","agent")),
        adresse         VARCHAR(100),
        superieur_id    INTEGER REFERENCES personnel(personnel_id),
        bureau_id       INTEGER REFERENCES bureau(bureau_id),
        annee_integration INTEGER,
    photo_profil      TEXT
    );
    CREATE TABLE IF NOT EXISTS activite(
        activite_id   INTEGER PRIMARY KEY AUTOINCREMENT,
        nom           VARCHAR(100),
        type_activite VARCHAR(30),
        description   TEXT,
        dateDebut     DATE,
        dateFin       DATE,
        statut        VARCHAR(20) DEFAULT "Planifiee",
        bureau_id     INTEGER REFERENCES bureau(bureau_id),
        propose_par   INTEGER REFERENCES personnel(personnel_id),
        validee_par   INTEGER REFERENCES personnel(personnel_id)
    );
    CREATE TABLE IF NOT EXISTS proposition_activite(
        prop_id       INTEGER PRIMARY KEY AUTOINCREMENT,
        description   TEXT,
        type_activite VARCHAR(30),
        date_prop     DATE,
        statut        VARCHAR(20) DEFAULT "En attente",
        propose_par   INTEGER REFERENCES personnel(personnel_id),
        destine_a     INTEGER REFERENCES personnel(personnel_id),
        commentaire_reponse TEXT
    );
    CREATE TABLE IF NOT EXISTS performance(
        performance_id INTEGER PRIMARY KEY AUTOINCREMENT,
        efficacite     INTEGER CHECK(efficacite BETWEEN 0 AND 100),
        note           INTEGER CHECK(note BETWEEN 0 AND 20),
        prime          VARCHAR(50),
        commentaire    TEXT,
        personnel_id   INTEGER REFERENCES personnel(personnel_id),
        evalue_par     INTEGER REFERENCES personnel(personnel_id),
        date_eval      DATE,
        mois           INTEGER,
        annee          INTEGER
    );
    CREATE TABLE IF NOT EXISTS employe_mois(
        em_id          INTEGER PRIMARY KEY AUTOINCREMENT,
        personnel_id   INTEGER REFERENCES personnel(personnel_id),
        mois           INTEGER NOT NULL,
        annee          INTEGER NOT NULL,
        note_finale    REAL,
        motif          TEXT,
        designe_par    INTEGER REFERENCES personnel(personnel_id),
        date_designation DATE,
        UNIQUE(mois, annee)
    );
    CREATE TABLE IF NOT EXISTS tache(
        tache_id     INTEGER PRIMARY KEY AUTOINCREMENT,
        libelle      VARCHAR(100),
        description  TEXT,
        dateDebut    DATE,
        dateFin      DATE,
        statut       VARCHAR(30) DEFAULT "Non demarre",
        priorite     VARCHAR(20) DEFAULT "Normale",
        activite_id  INTEGER REFERENCES activite(activite_id),
        performance_id INTEGER REFERENCES performance(performance_id)
    );
    CREATE TABLE IF NOT EXISTS affecter(
        affect_id        INTEGER PRIMARY KEY AUTOINCREMENT,
        tache_id         INTEGER REFERENCES tache(tache_id),
        personnel_id     INTEGER REFERENCES personnel(personnel_id),
        role_affect      TEXT DEFAULT "Executant",
        date_affectation DATE,
        date_retrait     DATE,
        actif            INTEGER DEFAULT 1,
        assigne_par      INTEGER REFERENCES personnel(personnel_id)
    );
    CREATE TABLE IF NOT EXISTS notification(
        notification_id INTEGER PRIMARY KEY AUTOINCREMENT,
        titre           TEXT NOT NULL,
        description     TEXT NOT NULL,
        type_notif      VARCHAR(30) DEFAULT "info",
        dateEnvoie      DATETIME DEFAULT CURRENT_TIMESTAMP,
        statut          VARCHAR(20) DEFAULT "Non lue",
        tache_id        INTEGER REFERENCES tache(tache_id),
        destinataire_id INTEGER REFERENCES personnel(personnel_id)
    );
    CREATE TABLE IF NOT EXISTS signalement(
        signalement_id INTEGER PRIMARY KEY AUTOINCREMENT,
        description    TEXT,
        dateEnvoie     DATE,
        statut         VARCHAR(30) DEFAULT "Ouvert",
        reponse        TEXT,
        personnel_id   INTEGER REFERENCES personnel(personnel_id),
        tache_id       INTEGER REFERENCES tache(tache_id),
        destine_a      INTEGER REFERENCES personnel(personnel_id)
    );
    CREATE TABLE IF NOT EXISTS idee(
        idee_id     INTEGER PRIMARY KEY AUTOINCREMENT,
        description TEXT,
        dateRecue   DATE,
        statut      VARCHAR(30) DEFAULT "Soumise",
        activite_id INTEGER REFERENCES activite(activite_id)
    );
    CREATE TABLE IF NOT EXISTS proposition_idee(
        idee_id      INTEGER REFERENCES idee(idee_id),
        personnel_id INTEGER REFERENCES personnel(personnel_id),
        PRIMARY KEY(idee_id, personnel_id)
    );
    CREATE TABLE IF NOT EXISTS avis(
        avis_id      INTEGER PRIMARY KEY AUTOINCREMENT,
        commentaire  TEXT,
        note         INTEGER CHECK(note BETWEEN 1 AND 5),
        dateEnvoie   DATE,
        personnel_id INTEGER REFERENCES personnel(personnel_id)
    );
    CREATE TABLE IF NOT EXISTS compteRendu(
        cr_id        INTEGER PRIMARY KEY AUTOINCREMENT,
        dateRendue   DATE,
        contenu      TEXT NOT NULL,
        statut       VARCHAR(20) DEFAULT "Brouillon",
        personnel_id INTEGER REFERENCES personnel(personnel_id),
        tache_id     INTEGER REFERENCES tache(tache_id)
    );
    CREATE TABLE IF NOT EXISTS telepaiement(
        tp_id            INTEGER PRIMARY KEY AUTOINCREMENT,
        reference        TEXT UNIQUE NOT NULL,
        contribuable_nom TEXT NOT NULL,
        contribuable_nif TEXT,
        type_impot       TEXT NOT NULL,
        montant          REAL NOT NULL,
        montant_paye     REAL DEFAULT 0,
        statut           VARCHAR(30) DEFAULT "En attente",
        mode_paiement    TEXT,
        date_echeance    DATE,
        date_paiement    DATE,
        bureau_id        INTEGER REFERENCES bureau(bureau_id),
        agent_id         INTEGER REFERENCES personnel(personnel_id),
        notes            TEXT
    );
    ''');
    conn.commit(); conn.close()
    print('✅ Base de données créée avec succès')

def seed():
    conn = get_db(); c = conn.cursor()
    if c.execute('SELECT COUNT(*) FROM personnel').fetchone()[0] > 0:
        conn.close(); print('ℹ️  Données déjà présentes, seed ignoré'); return

    # ── 5 BUREAUX RÉELS ──
    bureaux = [
        ('Gestion des Contribuables et Services','GCS','Accueil, immatriculation, assistance et gestion des dossiers des contribuables'),
        ('Recouvrement','REC','Recouvrement des créances fiscales, mise en demeure, contraintes'),
        ('Bureau des Domaines','DOM','Gestion du domaine national, titres fonciers, concessions'),
        ('Conservation Foncière','COF','Conservation des titres de propriété, mutations, hypothèques'),
        ('Cadastre','CAD','Levés cadastraux, plans fonciers, fiscalité immobilière'),
    ]
    c.executemany('INSERT INTO bureau(nom,code,description) VALUES(?,?,?)', bureaux)

    pw = h('admin123')

    # ── CHEF DE CENTRE ──
    c.execute("INSERT INTO personnel(nom,prenom,email,mot_de_passe,fonction,role,bureau_id,telephone,annee_integration) VALUES(?,?,?,?,?,?,?,?,?)",
              ('DIALLO','PENDA','chef.centre@terrafisc.sn',pw,'Directeur du Centre des Impôts','chef_centre',1,'771000001',2015))
    cc = c.lastrowid

    # ── CHEFS DE BUREAU (1 par bureau) ──
    chefs_data = [
        ('DIOP','ABDOUL','cb.gcs@terrafisc.sn','Chef du Bureau GCS','chef_bureau',1),
        ('TOURE','Ibrahima','cb.rec@terrafisc.sn','Chef du Bureau Recouvrement','chef_bureau',2),
        ('MBENGUE','ABDOURAHMANE','cb.dom@terrafisc.sn','Chef du Bureau des Domaines','chef_bureau',3),
        ('DIOUF','ELHADJI IBRAHIMA','cb.cof@terrafisc.sn','Chef de la Conservation Foncière','chef_bureau',4),
        ('SECK','FALLOU','cb.cad@terrafisc.sn','Chef du Cadastre','chef_bureau',5),
    ]
    cb_ids = []
    for i,(nom,prenom,email,fonc,role,bid) in enumerate(chefs_data):
        c.execute("INSERT INTO personnel(nom,prenom,email,mot_de_passe,fonction,role,bureau_id,superieur_id,telephone,annee_integration) VALUES(?,?,?,?,?,?,?,?,?,?)",
                  (nom,prenom,email,pw,fonc,role,bid,cc,f'77200000{i+1}',2017+i))
        cb_ids.append(c.lastrowid)

    # ── CONTRÔLEURS (10 par bureau) ──
    ctrl_data = [
        ('KA','AMINATA','ctrl.gcs1@terrafisc.sn','Contrôleur',1,cb_ids[0]),
        ('DIAW','AMI','ctrl.gcs2@terrafisc.sn','Inspecteur',1,cb_ids[0]),
        ('KEBE','OUSMANE','ctrl.rec1@terrafisc.sn','Contrôleur',2,cb_ids[1]),
        ('DIALLO','MOUHAMADOUL MOUKHTAR','ctrl.rec2@terrafisc.sn','Inspecteur',2,cb_ids[1]),
        ('SOW','Pape','ctrl.dom1@terrafisc.sn','Contrôleur',3,cb_ids[2]),
        ('FAYE','Aminata','ctrl.dom2@terrafisc.sn','Inspecteur',3,cb_ids[2]),
        ('DIOP','Serigne','ctrl.cof1@terrafisc.sn','Contrôleur',4,cb_ids[3]),
        ('CISSE','Adja','ctrl.cof2@terrafisc.sn','Inspecteur',4,cb_ids[3]),
        ('TOURE','Boubacar','ctrl.cad1@terrafisc.sn','Contrôleur',5,cb_ids[4]),
        ('LY','Marième','ctrl.cad2@terrafisc.sn','Inspecteur',5,cb_ids[4]),
    ]
    ctrl_ids = []
    for i,(nom,prenom,email,fonc,bid,sup) in enumerate(ctrl_data):
        c.execute("INSERT INTO personnel(nom,prenom,email,mot_de_passe,fonction,role,bureau_id,superieur_id,telephone,annee_integration) VALUES(?,?,?,?,?,?,?,?,?,?)",
                  (nom,prenom,email,pw,fonc,'controleur',bid,sup,f'77300{i:04d}',2019+i%4))
        ctrl_ids.append(c.lastrowid)

    # ── AGENTS (30 par bureau) ──
    agents_data = [
        ('NGOM','Awa','agent.gcs1@terrafisc.sn','Agent d\'accueil',1,ctrl_ids[0]),
        ('BADJI','Lamine','agent.gcs2@terrafisc.sn','Agent de gestion',1,ctrl_ids[0]),
        ('BODIAN','Seynabou','agent.gcs3@terrafisc.sn','Téléconseiller',1,ctrl_ids[1]),
        ('DIEME','Aliou','agent.rec1@terrafisc.sn','Agent de recouvrement',2,ctrl_ids[2]),
        ('CAMARA','Khady','agent.rec2@terrafisc.sn','Agent de recouvrement',2,ctrl_ids[2]),
        ('MENDY','Ismaila','agent.rec3@terrafisc.sn','Agent de recouvrement',2,ctrl_ids[3]),
        ('BASSENE','Ndéye','agent.dom1@terrafisc.sn','Agent domanial',3,ctrl_ids[4]),
        ('DIATTA','Omar','agent.dom2@terrafisc.sn','Agent domanial',3,ctrl_ids[4]),
        ('MANGA','Madeleine','agent.dom3@terrafisc.sn','Agent domanial',3,ctrl_ids[5]),
        ('GOMIS','Pascal','agent.cof1@terrafisc.sn','Agent foncier',4,ctrl_ids[6]),
        ('SAMBOU','Bernadette','agent.cof2@terrafisc.sn','Agent foncier',4,ctrl_ids[6]),
        ('TENDENG','Joseph','agent.cof3@terrafisc.sn','Agent foncier',4,ctrl_ids[7]),
        ('SANE','Ousmane','agent.cad1@terrafisc.sn','Géomètre',5,ctrl_ids[8]),
        ('BADIANE','Célestine','agent.cad2@terrafisc.sn','Technicien cadastral',5,ctrl_ids[8]),
        ('COLY','Augustin','agent.cad3@terrafisc.sn','Technicien cadastral',5,ctrl_ids[9]),
    ]
    agent_ids = []
    for i,(nom,prenom,email,fonc,bid,sup) in enumerate(agents_data):
        c.execute("INSERT INTO personnel(nom,prenom,email,mot_de_passe,fonction,role,bureau_id,superieur_id,telephone,annee_integration) VALUES(?,?,?,?,?,?,?,?,?,?)",
                  (nom,prenom,email,pw,fonc,'agent',bid,sup,f'77400{i:04d}',2020+i%5))
        agent_ids.append(c.lastrowid)

    # ── ACTIVITÉS PAR BUREAU ──
    activites_data = [
        ('Campagne immatriculation NIF 2025','Campagne','Sensibilisation et immatriculation des contribuables non identifiés','2025-04-01','2025-06-30','En cours',1),
        ('Audit des dossiers GCS Q1','Audit','Vérification et mise à jour des dossiers contribuables','2025-03-01','2025-03-31','Terminee',1),
        ('Opération recouvrement arrières T1','Mission','Recouvrement des créances fiscales du 1er trimestre','2025-04-10','2025-05-31','En cours',2),
        ('Formation procédures de mise en demeure','Formation','Formation des agents sur les nouvelles procédures','2025-05-15','2025-05-17','Planifiee',2),
        ('Révision des titres fonciers zone industrielle','Mission','Mise à jour du registre des titres fonciers','2025-04-01','2025-07-31','En cours',3),
        ('Atelier foncier rural','Atelier','Sensibilisation sur la sécurisation foncière rurale','2025-06-01','2025-06-03','Planifiee',3),
        ('Conservation titres quartier Almadies','Mission','Traitement des mutations en attente','2025-04-15','2025-05-30','En cours',4),
        ('Informatisation registre hypothèques','Projet','Numérisation des actes hypothécaires','2025-01-01','2025-12-31','En cours',4),
        ('Levé cadastral zone périurbaine','Mission','Mise à jour du plan cadastral','2025-05-01','2025-08-31','Planifiee',5),
        ('Mise à jour base fiscalité immobilière','Mission','Actualisation des valeurs locatives','2025-04-01','2025-06-30','En cours',5),
    ]
    act_ids = []
    for t in activites_data:
        c.execute('INSERT INTO activite(nom,type_activite,description,dateDebut,dateFin,statut,bureau_id) VALUES(?,?,?,?,?,?,?)',t)
        act_ids.append(c.lastrowid)

    # ── TÂCHES ──
    taches_data = [
        ('Préparation formulaires NIF','Imprimer et distribuer les formulaires','2025-04-01','2025-04-05','Terminee','Haute',act_ids[0]),
        ('Saisie des nouvelles immatriculations','Saisir les données dans le système','2025-04-06','2025-06-30','En cours','Normale',act_ids[0]),
        ('Rapport audit Q1 GCS','Rédiger le rapport final de l\'audit','2025-03-25','2025-03-31','Terminee','Haute',act_ids[1]),
        ('Identification créanciers défaillants','Lister les contribuables en arriéré','2025-04-10','2025-04-20','Terminee','Haute',act_ids[2]),
        ('Émission mises en demeure','Préparer et envoyer les mises en demeure','2025-04-20','2025-05-10','En cours','Haute',act_ids[2]),
        ('TDR formation mise en demeure','Rédiger les termes de référence','2025-04-20','2025-04-30','En cours','Normale',act_ids[3]),
        ('Convocations formation','Envoyer les convocations aux participants','2025-05-01','2025-05-10','Non demarre','Normale',act_ids[3]),
        ('Inventaire titres fonciers zone ind.','Dresser la liste des titres concernés','2025-04-01','2025-04-15','Terminee','Haute',act_ids[4]),
        ('Mise à jour registre foncier','Actualiser le registre informatisé','2025-04-15','2025-07-31','En cours','Normale',act_ids[4]),
        ('Traitement mutations en attente','Traiter les dossiers de mutation','2025-04-15','2025-05-30','En cours','Haute',act_ids[6]),
        ('Scan actes hypothécaires','Numériser les actes du registre A','2025-01-15','2025-06-30','En cours','Normale',act_ids[7]),
        ('Actualisation valeurs locatives','Mettre à jour les données fiscales','2025-04-01','2025-06-30','En cours','Haute',act_ids[9]),
    ]
    tache_ids = []
    for t in taches_data:
        c.execute('INSERT INTO tache(libelle,description,dateDebut,dateFin,statut,priorite,activite_id) VALUES(?,?,?,?,?,?,?)',t)
        tache_ids.append(c.lastrowid)

    # ── AFFECTATIONS ──
    affects = [
        (tache_ids[0],agent_ids[0],'Principal','2025-04-01',ctrl_ids[0]),
        (tache_ids[1],agent_ids[1],'Executant','2025-04-06',ctrl_ids[0]),
        (tache_ids[1],agent_ids[2],'Executant','2025-04-06',ctrl_ids[0]),
        (tache_ids[2],agent_ids[1],'Principal','2025-03-20',ctrl_ids[1]),
        (tache_ids[3],agent_ids[3],'Principal','2025-04-10',ctrl_ids[2]),
        (tache_ids[4],agent_ids[4],'Executant','2025-04-20',ctrl_ids[2]),
        (tache_ids[4],agent_ids[5],'Executant','2025-04-20',ctrl_ids[3]),
        (tache_ids[5],agent_ids[6],'Principal','2025-04-20',ctrl_ids[4]),
        (tache_ids[7],agent_ids[6],'Principal','2025-04-01',ctrl_ids[4]),
        (tache_ids[8],agent_ids[7],'Executant','2025-04-15',ctrl_ids[5]),
        (tache_ids[9],agent_ids[9],'Principal','2025-04-15',ctrl_ids[6]),
        (tache_ids[10],agent_ids[10],'Executant','2025-01-15',ctrl_ids[7]),
        (tache_ids[11],agent_ids[13],'Principal','2025-04-01',ctrl_ids[9]),
    ]
    for (tid,pid,role,dt,sup) in affects:
        c.execute('INSERT INTO affecter(tache_id,personnel_id,role_affect,date_affectation,actif,assigne_par) VALUES(?,?,?,?,1,?)',(tid,pid,role,dt,sup))
        c.execute("INSERT INTO notification(titre,description,type_notif,tache_id,destinataire_id) VALUES(?,?,?,?,?)",
                  ('Nouvelle tâche assignée','Vous avez une nouvelle tâche. Consultez votre tableau de bord.','tache',tid,pid))

    # ── PERFORMANCES ──
    perfs = [
        (90,18,'75000 FCFA','Excellent travail sur la campagne NIF',agent_ids[0],ctrl_ids[0],'2025-03-31',3,2025),
        (82,16,'50000 FCFA','Bonne rigueur documentaire',agent_ids[1],ctrl_ids[0],'2025-03-31',3,2025),
        (95,19,'100000 FCFA','Performance exceptionnelle sur le recouvrement',agent_ids[3],ctrl_ids[2],'2025-03-31',3,2025),
        (78,15,None,'Résultats satisfaisants, à améliorer',agent_ids[4],ctrl_ids[2],'2025-03-31',3,2025),
        (88,17,'75000 FCFA','Très bonne gestion des titres fonciers',agent_ids[6],ctrl_ids[4],'2025-03-31',3,2025),
        (70,14,None,'Quelques retards à corriger',agent_ids[7],ctrl_ids[5],'2025-03-31',3,2025),
        (92,18,'75000 FCFA','Excellent sur les mutations foncières',agent_ids[9],ctrl_ids[6],'2025-03-31',3,2025),
        (85,17,'50000 FCFA','Bon travail sur la numérisation',agent_ids[13],ctrl_ids[9],'2025-03-31',3,2025),
    ]
    perf_ids = []
    for p in perfs:
        c.execute('INSERT INTO performance(efficacite,note,prime,commentaire,personnel_id,evalue_par,date_eval,mois,annee) VALUES(?,?,?,?,?,?,?,?,?)',p)
        perf_ids.append(c.lastrowid)

    # ── EMPLOYÉ DU MOIS ── (3 derniers mois)
    em_data = [
        (agent_ids[3], 1, 2025, 18.5, 'Recouvrement exceptionnel : 98% du quota atteint, zéro réclamation', cb_ids[1]),
        (agent_ids[0], 2, 2025, 17.8, 'Meilleure performance sur les immatriculations NIF du mois', cb_ids[0]),
        (agent_ids[9], 3, 2025, 18.0, 'Excellence dans le traitement des mutations et délais respectés', cb_ids[3]),
    ]
    for (pid, mois, annee, note, motif, desig) in em_data:
        c.execute('INSERT INTO employe_mois(personnel_id,mois,annee,note_finale,motif,designe_par,date_designation) VALUES(?,?,?,?,?,?,?)',
                  (pid,mois,annee,note,motif,desig,'2025-'+str(mois).zfill(2)+'-28'))

    # ── TÉLÉPAIEMENTS ──
    tp_data = [
        ('TP2025-001','SOCOCIM Industries','SN000001','Impôt sur les Sociétés',45000000,45000000,'Paye','Virement bancaire','2025-03-31','2025-03-28',1,agent_ids[0],None),
        ('TP2025-002','AUCHAN Sénégal','SN000002','TVA mensuelle',12500000,12500000,'Paye','Mobile Money (Wave)','2025-04-20','2025-04-18',1,agent_ids[1],None),
        ('TP2025-003','Groupe Maurel & Prom','SN000003','Taxe foncière',3200000,0,'En attente',None,'2025-04-30',None,5,agent_ids[13],'Contribuable en cours de constitution de dossier'),
        ('TP2025-004','SENELEC','SN000004','Acompte IS',18000000,18000000,'Paye','Virement BCEAO','2025-03-31','2025-03-30',2,agent_ids[3],None),
        ('TP2025-005','Air Sénégal','SN000005','Taxe sur les salaires',2800000,1400000,'Partiel','Mobile Money (Orange)','2025-04-15','2025-04-14',2,agent_ids[4],'Solde restant dû : 1 400 000 FCFA'),
        ('TP2025-006','SDE (Société des Eaux)','SN000006','TVA mensuelle',8900000,0,'En retard',None,'2025-04-10',None,2,agent_ids[5],'Mise en demeure émise le 15/04/2025'),
        ('TP2025-007','CBAO Banque','SN000007','Taxe foncière TF n°1234',950000,950000,'Paye','Carte bancaire','2025-03-15','2025-03-14',4,agent_ids[9],None),
        ('TP2025-008','Résidence Les Almadies','SN000008','Contribution foncière bâtie',2100000,0,'En attente',None,'2025-05-31',None,4,agent_ids[10],'Dossier complet reçu'),
        ('TP2025-009','Cabinet Géomètre Diop','SN000009','Droits de bornage',350000,350000,'Paye','Mobile Money (Wave)','2025-04-01','2025-04-01',5,agent_ids[13],None),
        ('TP2025-010','Lotissement SICAP','SN000010','Taxe de plus-value immobilière',5600000,5600000,'Paye','Virement bancaire','2025-04-05','2025-04-04',3,agent_ids[6],None),
    ]
    for tp in tp_data:
        c.execute('INSERT INTO telepaiement(reference,contribuable_nom,contribuable_nif,type_impot,montant,montant_paye,statut,mode_paiement,date_echeance,date_paiement,bureau_id,agent_id,notes) VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?)', tp)

    # ── SIGNALEMENTS ──
    c.execute("INSERT INTO signalement(description,dateEnvoie,statut,personnel_id,tache_id,destine_a) VALUES(?,?,?,?,?,?)",
              ('Manque de données dans le système pour 3 contribuables inscrits. Impossible de finaliser la saisie.','2025-04-22','Ouvert',agent_ids[1],tache_ids[1],ctrl_ids[0]))
    c.execute("INSERT INTO signalement(description,dateEnvoie,statut,personnel_id,tache_id,destine_a) VALUES(?,?,?,?,?,?)",
              ('Contribuable TP2025-006 (SDE) non joignable. Risque de dépassement de délai.','2025-04-20','Ouvert',agent_ids[5],tache_ids[4],ctrl_ids[3]))

    # ── PROPOSITIONS ──
    c.execute("INSERT INTO proposition_activite(description,type_activite,date_prop,statut,propose_par,destine_a) VALUES(?,?,?,?,?,?)",
              ('Organiser une journée portes ouvertes pour les nouveaux contribuables sur la procédure d\'immatriculation en ligne.','Atelier','2025-04-18','En attente',ctrl_ids[0],cb_ids[0]))
    c.execute("INSERT INTO proposition_activite(description,type_activite,date_prop,statut,propose_par,destine_a) VALUES(?,?,?,?,?,?)",
              ('Mettre en place un tableau de suivi partagé pour les mises en demeure en cours afin d\'éviter les doublons.','Mission','2025-04-20','Acceptee',ctrl_ids[2],cb_ids[1]))

    # ── AVIS ──
    c.execute("INSERT INTO avis(commentaire,note,dateEnvoie,personnel_id) VALUES(?,?,?,?)",('Interface claire, la section télépaiement est très utile.',5,'2025-04-25',agent_ids[0]))
    c.execute("INSERT INTO avis(commentaire,note,dateEnvoie,personnel_id) VALUES(?,?,?,?)",('Ajouter un export Excel des télépaiements.',4,'2025-04-26',agent_ids[3]))
    c.execute("INSERT INTO avis(commentaire,note,dateEnvoie,personnel_id) VALUES(?,?,?,?)",('Le module employé du mois motive vraiment l\'équipe !',5,'2025-04-27',agent_ids[9]))

    conn.commit(); conn.close()
    print('✅ Données insérées avec succès\n')
    print('━'*58)
    print('📋 COMPTES DE CONNEXION (mot de passe : admin123)')
    print('━'*58)
    print('  chef.centre@terrafisc.sn   →  Chef de Centre')
    print('  cb.gcs@terrafisc.sn        →  Chef Bureau GCS')
    print('  cb.rec@terrafisc.sn        →  Chef Bureau Recouvrement')
    print('  cb.dom@terrafisc.sn        →  Chef Bureau Domaines')
    print('  cb.cof@terrafisc.sn        →  Chef Conservation Foncière')
    print('  cb.cad@terrafisc.sn        →  Chef Cadastre')
    print('  ctrl.gcs1@terrafisc.sn     →  Contrôleur GCS')
    print('  ctrl.rec1@terrafisc.sn     →  Contrôleur Recouvrement')
    print('  agent.gcs1@terrafisc.sn    →  Agent GCS')
    print('  agent.rec1@terrafisc.sn    →  Agent Recouvrement')
    print('━'*58)

init_db()
seed()


✅ Base de données créée avec succès
✅ Données insérées avec succès

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📋 COMPTES DE CONNEXION (mot de passe : admin123)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  chef.centre@terrafisc.sn   →  Chef de Centre
  cb.gcs@terrafisc.sn        →  Chef Bureau GCS
  cb.rec@terrafisc.sn        →  Chef Bureau Recouvrement
  cb.dom@terrafisc.sn        →  Chef Bureau Domaines
  cb.cof@terrafisc.sn        →  Chef Conservation Foncière
  cb.cad@terrafisc.sn        →  Chef Cadastre
  ctrl.gcs1@terrafisc.sn     →  Contrôleur GCS
  ctrl.rec1@terrafisc.sn     →  Contrôleur Recouvrement
  agent.gcs1@terrafisc.sn    →  Agent GCS
  agent.rec1@terrafisc.sn    →  Agent Recouvrement
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## 🚀 Cellule 3 — Application Flask

In [4]:
from flask import Flask, request, jsonify, render_template_string, session
from flask_cors import CORS
import sqlite3, threading, hashlib, json
from datetime import date, datetime

app = Flask(__name__)
app.secret_key = 'terrafisc_2025_secret'
CORS(app, supports_credentials=True)

def h(p): return hashlib.sha256(p.encode()).hexdigest()

def gen_email(nom, prenom):
    import unicodedata, re as _re
    def clean(s):
        s = unicodedata.normalize('NFD', s or '')
        s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
        return _re.sub(r'[^a-z0-9]', '', s.lower())
    n = clean(nom); p = clean(prenom)
    base = f"{p}.{n}@terrafisc.sn" if p else f"{n}@terrafisc.sn"
    # Check uniqueness
    existing = qry("SELECT email FROM personnel WHERE email LIKE ?", (f"{p}.{n}%@terrafisc.sn",))
    if not any(r['email'] == base for r in existing):
        return base
    i = 2
    while True:
        candidate = f"{p}.{n}{i}@terrafisc.sn"
        if not any(r['email'] == candidate for r in existing):
            return candidate
        i += 1

def qry(sql, params=(), one=False):
    conn = get_db()
    rows = [dict(r) for r in conn.execute(sql, params).fetchall()]
    conn.close()
    return rows[0] if (one and rows) else (None if (one and not rows) else rows)

def exe(sql, params=()):
    conn = get_db(); cur = conn.execute(sql, params); conn.commit(); lid = cur.lastrowid; conn.close(); return lid

def notif(titre, desc, type_n, dest_id, tache_id=None):
    if dest_id:
        exe('INSERT INTO notification(titre,description,type_notif,tache_id,destinataire_id) VALUES(?,?,?,?,?)',
            (titre, desc, type_n, tache_id, dest_id))

def uid(): return session.get('user_id')
def role(): return session.get('role')
def auth(): return 'user_id' in session

MOIS_FR = ['','Janvier','Février','Mars','Avril','Mai','Juin','Juillet','Août','Septembre','Octobre','Novembre','Décembre']

# ═══════════════════════════════════════════════════════
#  AUTH
# ═══════════════════════════════════════════════════════
@app.route('/api/login', methods=['POST'])
def login():
    d = request.json
    u = qry('SELECT * FROM personnel WHERE email=? AND mot_de_passe=?', (d['email'], h(d['password'])), one=True)
    if not u: return jsonify({'error': 'Email ou mot de passe incorrect'}), 401
    session['user_id'] = u['personnel_id']
    session['role'] = u['role']
    u.pop('mot_de_passe', None)
    return jsonify(u)

@app.route('/api/logout', methods=['POST'])
def logout():
    session.clear(); return jsonify({'ok': True})

@app.route('/api/change-password', methods=['POST'])
def change_password():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    d = request.json
    u = qry('SELECT mot_de_passe FROM personnel WHERE personnel_id=?', (uid(),), one=True)
    if not u or u['mot_de_passe'] != h(d['old_password']):
        return jsonify({'error': 'Mot de passe actuel incorrect'}), 403
    exe('UPDATE personnel SET mot_de_passe=? WHERE personnel_id=?', (h(d['new_password']), uid()))
    return jsonify({'ok': True})

@app.route('/api/personnel/<int:pid>', methods=['GET'])
def get_personnel_by_id(pid):
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    row = qry('SELECT p.*,b.nom bureau_nom FROM personnel p LEFT JOIN bureau b ON p.bureau_id=b.bureau_id WHERE p.personnel_id=?', (pid,), one=True)
    if row: row.pop('mot_de_passe', None)
    return jsonify(row or {})

# ═══════════════════════════════════════════════════════
#  DASHBOARD
# ═══════════════════════════════════════════════════════
@app.route('/api/dashboard')
def dashboard():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    r = role(); u = uid()
    nb_notifs = qry("SELECT COUNT(*) c FROM notification WHERE destinataire_id=? AND statut='Non lue'", (u,), one=True)['c']

    if r == 'chef_centre':
        return jsonify({
            'nb_bureaux': qry('SELECT COUNT(*) c FROM bureau', one=True)['c'],
            'nb_personnel': qry('SELECT COUNT(*) c FROM personnel', one=True)['c'],
            'nb_activites': qry('SELECT COUNT(*) c FROM activite', one=True)['c'],
            'nb_taches': qry('SELECT COUNT(*) c FROM tache', one=True)['c'],
            'nb_notifs': nb_notifs,
            'taches_statut': qry('SELECT statut, COUNT(*) n FROM tache GROUP BY statut'),
            'activites_statut': qry('SELECT statut, COUNT(*) n FROM activite GROUP BY statut'),
            'bureaux_activites': qry('SELECT b.nom, COUNT(a.activite_id) n FROM bureau b LEFT JOIN activite a ON b.bureau_id=a.bureau_id GROUP BY b.nom'),
            'top_performances': qry('SELECT p.nom,p.prenom,pf.note,pf.efficacite,b.nom bureau_nom FROM performance pf JOIN personnel p ON pf.personnel_id=p.personnel_id JOIN bureau b ON p.bureau_id=b.bureau_id ORDER BY pf.note DESC LIMIT 8'),
            'employe_mois': qry('SELECT em.*,p.nom,p.prenom,b.nom bureau_nom FROM employe_mois em JOIN personnel p ON em.personnel_id=p.personnel_id JOIN bureau b ON p.bureau_id=b.bureau_id ORDER BY em.annee DESC,em.mois DESC LIMIT 3'),
            'tp_stats': qry("SELECT statut,COUNT(*) n,SUM(montant) total FROM telepaiement GROUP BY statut"),
        })
    elif r == 'chef_bureau':
        bid = qry('SELECT bureau_id FROM personnel WHERE personnel_id=?', (u,), one=True)['bureau_id']
        return jsonify({
            'nb_controleurs': qry('SELECT COUNT(*) c FROM personnel WHERE superieur_id=? AND role="controleur"', (u,), one=True)['c'],
            'nb_agents': qry('SELECT COUNT(*) c FROM personnel WHERE bureau_id=? AND role="agent"', (bid,), one=True)['c'],
            'nb_activites': qry('SELECT COUNT(*) c FROM activite WHERE bureau_id=?', (bid,), one=True)['c'],
            'nb_taches': qry('SELECT COUNT(*) c FROM tache t JOIN activite a ON t.activite_id=a.activite_id WHERE a.bureau_id=?', (bid,), one=True)['c'],
            'nb_notifs': nb_notifs,
            'taches_statut': qry('SELECT t.statut,COUNT(*) n FROM tache t JOIN activite a ON t.activite_id=a.activite_id WHERE a.bureau_id=? GROUP BY t.statut', (bid,)),
            'propositions_recues': qry("SELECT COUNT(*) c FROM proposition_activite WHERE destine_a=? AND statut='En attente'", (u,), one=True)['c'],
            'signalements_ouverts': qry("SELECT COUNT(*) c FROM signalement WHERE destine_a=? AND statut='Ouvert'", (u,), one=True)['c'],
            'employe_mois': qry('SELECT em.*,p.nom,p.prenom FROM employe_mois em JOIN personnel p ON em.personnel_id=p.personnel_id JOIN bureau b ON p.bureau_id=b.bureau_id WHERE p.bureau_id=? ORDER BY em.annee DESC,em.mois DESC LIMIT 3', (bid,)),
            'tp_bureau': qry('SELECT statut,COUNT(*) n,SUM(montant) total FROM telepaiement WHERE bureau_id=? GROUP BY statut', (bid,)),
        })
    elif r == 'controleur':
        return jsonify({
            'nb_agents': qry('SELECT COUNT(*) c FROM personnel WHERE superieur_id=?', (u,), one=True)['c'],
            'nb_taches_assignees': qry('SELECT COUNT(DISTINCT tache_id) c FROM affecter WHERE assigne_par=? AND actif=1', (u,), one=True)['c'],
            'nb_notifs': nb_notifs,
            'taches_statut': qry('SELECT t.statut,COUNT(*) n FROM tache t JOIN affecter af ON t.tache_id=af.tache_id WHERE af.assigne_par=? AND af.actif=1 GROUP BY t.statut', (u,)),
            'signalements_recus': qry("SELECT COUNT(*) c FROM signalement WHERE destine_a=? AND statut='Ouvert'", (u,), one=True)['c'],
            'mes_taches': qry('SELECT t.*,a.nom act_nom FROM tache t JOIN affecter af ON t.tache_id=af.tache_id JOIN activite a ON t.activite_id=a.activite_id WHERE af.personnel_id=? AND af.actif=1 ORDER BY t.dateFin', (u,)),
        })
    else:  # agent
        return jsonify({
            'nb_taches': qry('SELECT COUNT(*) c FROM affecter WHERE personnel_id=? AND actif=1', (u,), one=True)['c'],
            'nb_terminees': qry("SELECT COUNT(*) c FROM tache t JOIN affecter af ON t.tache_id=af.tache_id WHERE af.personnel_id=? AND af.actif=1 AND t.statut='Terminee'", (u,), one=True)['c'],
            'nb_notifs': nb_notifs,
            'ma_performance': qry('SELECT * FROM performance WHERE personnel_id=? ORDER BY date_eval DESC LIMIT 1', (u,), one=True),
            'mes_taches': qry('SELECT t.*,a.nom act_nom FROM tache t JOIN affecter af ON t.tache_id=af.tache_id JOIN activite a ON t.activite_id=a.activite_id WHERE af.personnel_id=? AND af.actif=1 ORDER BY t.dateFin', (u,)),
            'employe_mois_courant': qry('SELECT em.*,p.nom,p.prenom FROM employe_mois em JOIN personnel p ON em.personnel_id=p.personnel_id WHERE em.mois=? AND em.annee=?', (date.today().month, date.today().year), one=True),
        })

# ═══════════════════════════════════════════════════════
#  BUREAUX
# ═══════════════════════════════════════════════════════
@app.route('/api/bureaux', methods=['GET', 'POST'])
def bureaux():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    if request.method == 'POST':
        if role() != 'chef_centre': return jsonify({'error': 'Non autorisé'}), 403
        d = request.json
        lid = exe('INSERT INTO bureau(nom,code,description) VALUES(?,?,?)', (d['nom'], d.get('code'), d.get('description')))
        return jsonify({'bureau_id': lid}), 201
    return jsonify(qry('SELECT b.*, (SELECT COUNT(*) FROM personnel WHERE bureau_id=b.bureau_id) nb_pers, (SELECT COUNT(*) FROM activite WHERE bureau_id=b.bureau_id) nb_act FROM bureau b'))

# ═══════════════════════════════════════════════════════
#  PERSONNEL
# ═══════════════════════════════════════════════════════
@app.route('/api/personnel', methods=['GET', 'POST'])
def personnel_list():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'POST':
        if r not in ('chef_centre', 'chef_bureau'): return jsonify({'error': 'Non autorisé'}), 403
        d = request.json
        # Auto-generate email from nom/prenom
        email_auto = gen_email(d['nom'], d.get('prenom', ''))
        pwd_init = d.get('password', 'terrafisc2025')
        lid = exe('INSERT INTO personnel(nom,prenom,email,mot_de_passe,telephone,dateNaissance,fonction,role,adresse,superieur_id,bureau_id,annee_integration) VALUES(?,?,?,?,?,?,?,?,?,?,?,?)',
            (d['nom'],d.get('prenom'),email_auto,h(pwd_init),d.get('telephone'),d.get('dateNaissance'),d.get('fonction'),d['role'],d.get('adresse'),d.get('superieur_id'),d.get('bureau_id'),d.get('annee_integration',date.today().year)))
        notif('Bienvenue sur TERRAFISC', f"Vos identifiants : Email: {email_auto} | Mot de passe: {pwd_init}", 'info', lid)
        return jsonify({'personnel_id': lid, 'email': email_auto, 'password': pwd_init}), 201
    if r == 'chef_centre':
        rows = qry('SELECT p.*,b.nom bureau_nom,s.nom sup_nom FROM personnel p LEFT JOIN bureau b ON p.bureau_id=b.bureau_id LEFT JOIN personnel s ON p.superieur_id=s.personnel_id ORDER BY p.role,p.nom')
    elif r == 'chef_bureau':
        bid = qry('SELECT bureau_id FROM personnel WHERE personnel_id=?', (u,), one=True)['bureau_id']
        rows = qry('SELECT p.*,b.nom bureau_nom FROM personnel p LEFT JOIN bureau b ON p.bureau_id=b.bureau_id WHERE p.bureau_id=? ORDER BY p.role,p.nom', (bid,))
    elif r == 'controleur':
        rows = qry('SELECT p.*,b.nom bureau_nom FROM personnel p LEFT JOIN bureau b ON p.bureau_id=b.bureau_id WHERE p.superieur_id=?', (u,))
    else:
        rows = qry('SELECT p.*,b.nom bureau_nom FROM personnel p LEFT JOIN bureau b ON p.bureau_id=b.bureau_id WHERE p.personnel_id=?', (u,))
    for row in rows: row.pop('mot_de_passe', None)
    return jsonify(rows)

@app.route('/api/personnel/<int:pid>/photo', methods=['POST'])
def upload_photo(pid):
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    if uid() != pid and role() not in ('chef_centre', 'chef_bureau'):
        return jsonify({'error': 'Non autorisé'}), 403
    d = request.json
    exe('UPDATE personnel SET photo_profil=? WHERE personnel_id=?', (d.get('photo'), pid))
    return jsonify({'ok': True})

@app.route('/api/personnel/<int:pid>', methods=['PUT', 'DELETE'])
def personnel_detail(pid):
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    r = role(); u = uid()
    if request.method == 'DELETE':
        # Seul le chef de centre peut supprimer un personnel
        if r != 'chef_centre': return jsonify({'error': 'Non autorisé'}), 403
        exe('DELETE FROM personnel WHERE personnel_id=?', (pid,)); return jsonify({'ok': True})
    # PUT : chacun peut modifier ses propres infos; chef_centre/chef_bureau peuvent tout modifier
    if u != pid and r not in ('chef_centre', 'chef_bureau'):
        return jsonify({'error': 'Non autorisé'}), 403
    d = request.json
    # Champs modifiables par soi-même
    self_fields = ['nom', 'prenom', 'telephone', 'adresse', 'dateNaissance', 'photo_profil']
    # Champs admin uniquement
    admin_fields = ['email', 'fonction', 'role', 'superieur_id', 'bureau_id']
    sets, vals = [], []
    for f in self_fields:
        if f in d:
            sets.append(f'{f}=?'); vals.append(d[f])
    if r in ('chef_centre', 'chef_bureau'):
        for f in admin_fields:
            if f in d:
                sets.append(f'{f}=?'); vals.append(d[f])
    if not sets: return jsonify({'error': 'Rien à modifier'}), 400
    vals.append(pid)
    exe(f'UPDATE personnel SET {",".join(sets)} WHERE personnel_id=?', vals)
    return jsonify({'ok': True})

# ═══════════════════════════════════════════════════════
#  ACTIVITÉS
# ═══════════════════════════════════════════════════════
@app.route('/api/activites', methods=['GET', 'POST'])
def activites():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'POST':
        if r not in ('chef_centre', 'chef_bureau'): return jsonify({'error': 'Non autorisé'}), 403
        d = request.json
        lid = exe('INSERT INTO activite(nom,type_activite,description,dateDebut,dateFin,statut,bureau_id,propose_par) VALUES(?,?,?,?,?,?,?,?)',
            (d['nom'],d.get('type_activite'),d.get('description'),d.get('dateDebut'),d.get('dateFin'),d.get('statut','Planifiee'),d.get('bureau_id'),u))
        return jsonify({'activite_id': lid}), 201
    base = 'SELECT a.*,b.nom bureau_nom,(SELECT COUNT(*) FROM tache WHERE activite_id=a.activite_id) nb_taches,(SELECT COUNT(*) FROM tache WHERE activite_id=a.activite_id AND statut="Terminee") nb_ok FROM activite a LEFT JOIN bureau b ON a.bureau_id=b.bureau_id'
    if r == 'chef_centre':
        return jsonify(qry(base + ' ORDER BY a.dateDebut DESC'))
    elif r == 'chef_bureau':
        bid = qry('SELECT bureau_id FROM personnel WHERE personnel_id=?', (u,), one=True)['bureau_id']
        return jsonify(qry(base + ' WHERE a.bureau_id=? ORDER BY a.dateDebut DESC', (bid,)))
    else:
        return jsonify(qry(base + ' JOIN tache t ON a.activite_id=t.activite_id JOIN affecter af ON t.tache_id=af.tache_id WHERE af.personnel_id=? AND af.actif=1 GROUP BY a.activite_id', (u,)))

@app.route('/api/activites/<int:aid>', methods=['PUT', 'DELETE'])
def activite_detail(aid):
    if not auth() or role() not in ('chef_centre', 'chef_bureau'): return jsonify({'error': 'Non autorisé'}), 403
    if request.method == 'DELETE':
        exe('DELETE FROM activite WHERE activite_id=?', (aid,)); return jsonify({'ok': True})
    d = request.json
    exe('UPDATE activite SET nom=?,type_activite=?,description=?,dateDebut=?,dateFin=?,statut=? WHERE activite_id=?',
        (d.get('nom'),d.get('type_activite'),d.get('description'),d.get('dateDebut'),d.get('dateFin'),d.get('statut'),aid))
    return jsonify({'ok': True})

@app.route('/api/activites/<int:aid>/gantt')
def gantt(aid):
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    return jsonify({
        'activite': qry('SELECT * FROM activite WHERE activite_id=?', (aid,), one=True),
        'taches': qry('SELECT t.*,GROUP_CONCAT(p.nom||\'  \'||COALESCE(p.prenom,\'\')) agents FROM tache t LEFT JOIN affecter af ON t.tache_id=af.tache_id AND af.actif=1 LEFT JOIN personnel p ON af.personnel_id=p.personnel_id WHERE t.activite_id=? GROUP BY t.tache_id ORDER BY t.dateDebut', (aid,))
    })

# ═══════════════════════════════════════════════════════
#  TÂCHES
# ═══════════════════════════════════════════════════════
@app.route('/api/taches', methods=['GET', 'POST'])
def taches():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'POST':
        if r not in ('chef_bureau', 'controleur'): return jsonify({'error': 'Non autorisé'}), 403
        d = request.json
        lid = exe('INSERT INTO tache(libelle,description,dateDebut,dateFin,statut,priorite,activite_id) VALUES(?,?,?,?,?,?,?)',
            (d['libelle'],d.get('description'),d.get('dateDebut'),d.get('dateFin'),d.get('statut','Non demarre'),d.get('priorite','Normale'),d.get('activite_id')))
        return jsonify({'tache_id': lid}), 201
    base = 'SELECT t.*,a.nom act_nom,GROUP_CONCAT(p.nom||\'  \'||COALESCE(p.prenom,\'\')) agents FROM tache t LEFT JOIN activite a ON t.activite_id=a.activite_id LEFT JOIN affecter af ON t.tache_id=af.tache_id AND af.actif=1 LEFT JOIN personnel p ON af.personnel_id=p.personnel_id'
    if r == 'chef_centre':
        return jsonify(qry(base + ' GROUP BY t.tache_id ORDER BY t.dateFin'))
    elif r == 'chef_bureau':
        bid = qry('SELECT bureau_id FROM personnel WHERE personnel_id=?', (u,), one=True)['bureau_id']
        return jsonify(qry(base + ' WHERE a.bureau_id=? GROUP BY t.tache_id ORDER BY t.dateFin', (bid,)))
    elif r == 'controleur':
        return jsonify(qry(base + ' WHERE af.assigne_par=? OR af.personnel_id=? GROUP BY t.tache_id ORDER BY t.dateFin', (u, u)))
    else:
        return jsonify(qry('SELECT t.*,a.nom act_nom,af.role_affect FROM tache t JOIN affecter af ON t.tache_id=af.tache_id JOIN activite a ON t.activite_id=a.activite_id WHERE af.personnel_id=? AND af.actif=1 ORDER BY t.dateFin', (u,)))

@app.route('/api/taches/<int:tid>', methods=['PUT', 'DELETE'])
def tache_detail(tid):
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'DELETE':
        if r not in ('chef_bureau', 'controleur'): return jsonify({'error': 'Non autorisé'}), 403
        exe('DELETE FROM tache WHERE tache_id=?', (tid,)); return jsonify({'ok': True})
    d = request.json; old = qry('SELECT * FROM tache WHERE tache_id=?', (tid,), one=True)
    if r == 'agent':
        exe('UPDATE tache SET statut=? WHERE tache_id=?', (d.get('statut', old['statut']), tid))
    else:
        exe('UPDATE tache SET libelle=?,description=?,dateDebut=?,dateFin=?,statut=?,priorite=? WHERE tache_id=?',
            (d.get('libelle',old['libelle']),d.get('description',old['description']),d.get('dateDebut',old['dateDebut']),d.get('dateFin',old['dateFin']),d.get('statut',old['statut']),d.get('priorite',old['priorite']),tid))
    if d.get('statut') and d['statut'] != old.get('statut'):
        for ag in qry('SELECT personnel_id FROM affecter WHERE tache_id=? AND actif=1', (tid,)):
            notif('Mise à jour tâche', f'Tâche "{old["libelle"]}" → statut : {d["statut"]}', 'tache', ag['personnel_id'], tid)
    return jsonify({'ok': True})

@app.route('/api/taches/<int:tid>/affecter', methods=['POST'])
def affecter(tid):
    if not auth() or role() not in ('chef_centre', 'chef_bureau', 'controleur'): return jsonify({'error': 'Non autorisé'}), 403
    u = uid(); d = request.json; dest = int(d['personnel_id'])
    subs = [s['personnel_id'] for s in qry('SELECT personnel_id FROM personnel WHERE superieur_id=?', (u,))]
    if role() != 'chef_centre' and dest not in subs:
        return jsonify({'error': 'Vous ne pouvez affecter qu\'à vos subordonnés directs'}), 403
    exe('INSERT INTO affecter(tache_id,personnel_id,role_affect,date_affectation,actif,assigne_par) VALUES(?,?,?,?,1,?)',
        (tid, dest, d.get('role', 'Executant'), str(date.today()), u))
    tl = qry('SELECT libelle FROM tache WHERE tache_id=?', (tid,), one=True)
    notif(f'Nouvelle tâche assignée', f'Vous avez été affecté(e) à : "{tl["libelle"]}". Consultez votre tableau de bord.', 'tache', dest, tid)
    return jsonify({'ok': True})

@app.route('/api/taches/<int:tid>/retirer/<int:pid>', methods=['DELETE'])
def retirer(tid, pid):
    if not auth() or role() not in ('chef_centre', 'chef_bureau', 'controleur'): return jsonify({'error': 'Non autorisé'}), 403
    exe('UPDATE affecter SET actif=0,date_retrait=? WHERE tache_id=? AND personnel_id=?', (str(date.today()), tid, pid))
    notif('Retrait d\'affectation', 'Vous avez été retiré(e) d\'une tâche.', 'info', pid, tid)
    return jsonify({'ok': True})

@app.route('/api/taches/<int:tid>/historique')
def historique_tache(tid):
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    return jsonify(qry('SELECT af.*,p.nom||\'  \'||COALESCE(p.prenom,\'\') agent_nom,s.nom||\'  \'||COALESCE(s.prenom,\'\') sup_nom FROM affecter af JOIN personnel p ON af.personnel_id=p.personnel_id LEFT JOIN personnel s ON af.assigne_par=s.personnel_id WHERE af.tache_id=? ORDER BY af.date_affectation DESC', (tid,)))

# ═══════════════════════════════════════════════════════
#  PERFORMANCES
# ═══════════════════════════════════════════════════════
@app.route('/api/performances', methods=['GET', 'POST'])
def performances():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'POST':
        if r not in ('chef_bureau', 'controleur'): return jsonify({'error': 'Non autorisé'}), 403
        d = request.json; dest = int(d['personnel_id'])
        subs = [s['personnel_id'] for s in qry('SELECT personnel_id FROM personnel WHERE superieur_id=?', (u,))]
        if dest not in subs: return jsonify({'error': 'Vous ne pouvez évaluer que vos subordonnés'}), 403
        m = date.today().month; an = date.today().year
        lid = exe('INSERT INTO performance(efficacite,note,prime,commentaire,personnel_id,evalue_par,date_eval,mois,annee) VALUES(?,?,?,?,?,?,?,?,?)',
            (d.get('efficacite'),d.get('note'),d.get('prime'),d.get('commentaire'),dest,u,str(date.today()),m,an))
        notif('Nouvelle évaluation reçue', f'Vous avez reçu une évaluation pour {MOIS_FR[m]} {an}.', 'performance', dest)
        return jsonify({'performance_id': lid}), 201
    if r == 'chef_centre':
        return jsonify(qry('SELECT pf.*,p.nom,p.prenom,b.nom bureau_nom,e.nom eval_nom FROM performance pf JOIN personnel p ON pf.personnel_id=p.personnel_id JOIN bureau b ON p.bureau_id=b.bureau_id LEFT JOIN personnel e ON pf.evalue_par=e.personnel_id ORDER BY pf.date_eval DESC'))
    elif r == 'chef_bureau':
        bid = qry('SELECT bureau_id FROM personnel WHERE personnel_id=?', (u,), one=True)['bureau_id']
        return jsonify(qry('SELECT pf.*,p.nom,p.prenom,e.nom eval_nom FROM performance pf JOIN personnel p ON pf.personnel_id=p.personnel_id LEFT JOIN personnel e ON pf.evalue_par=e.personnel_id WHERE p.bureau_id=? ORDER BY pf.date_eval DESC', (bid,)))
    elif r == 'controleur':
        return jsonify(qry('SELECT pf.*,p.nom,p.prenom FROM performance pf JOIN personnel p ON pf.personnel_id=p.personnel_id WHERE pf.evalue_par=? ORDER BY pf.date_eval DESC', (u,)))
    else:
        return jsonify(qry('SELECT pf.*,e.nom eval_nom FROM performance pf LEFT JOIN personnel e ON pf.evalue_par=e.personnel_id WHERE pf.personnel_id=? ORDER BY pf.date_eval DESC', (u,)))

# ═══════════════════════════════════════════════════════
#  EMPLOYÉ DU MOIS
# ═══════════════════════════════════════════════════════
@app.route('/api/employe-mois', methods=['GET', 'POST'])
def employe_mois():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    if request.method == 'POST':
        if role() not in ('chef_centre', 'chef_bureau'): return jsonify({'error': 'Non autorisé'}), 403
        d = request.json
        m = int(d.get('mois', date.today().month))
        an = int(d.get('annee', date.today().year))
        # Remplacer si déjà existant pour ce mois/année
        existing = qry('SELECT em_id FROM employe_mois WHERE mois=? AND annee=?', (m, an), one=True)
        if existing:
            exe('UPDATE employe_mois SET personnel_id=?,note_finale=?,motif=?,designe_par=?,date_designation=? WHERE mois=? AND annee=?',
                (d['personnel_id'],d.get('note_finale'),d.get('motif'),uid(),str(date.today()),m,an))
        else:
            exe('INSERT INTO employe_mois(personnel_id,mois,annee,note_finale,motif,designe_par,date_designation) VALUES(?,?,?,?,?,?,?)',
                (d['personnel_id'],m,an,d.get('note_finale'),d.get('motif'),uid(),str(date.today())))
        notif('🏆 Employé du mois !', f'Félicitations ! Vous êtes désigné(e) Employé du mois de {MOIS_FR[m]} {an}.', 'employe_mois', int(d['personnel_id']))
        return jsonify({'ok': True}), 201
    return jsonify(qry('SELECT em.*,p.nom,p.prenom,p.fonction,b.nom bureau_nom,d.nom designateur_nom FROM employe_mois em JOIN personnel p ON em.personnel_id=p.personnel_id JOIN bureau b ON p.bureau_id=b.bureau_id LEFT JOIN personnel d ON em.designe_par=d.personnel_id ORDER BY em.annee DESC,em.mois DESC'))

# ═══════════════════════════════════════════════════════
#  TÉLÉPAIEMENT
# ═══════════════════════════════════════════════════════
@app.route('/api/telepaiements', methods=['GET', 'POST'])
def telepaiements():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'POST':
        if r not in ('chef_bureau', 'controleur', 'agent'): return jsonify({'error': 'Non autorisé'}), 403
        d = request.json
        ref = f'TP{date.today().year}-{str(exe("SELECT COUNT(*) c FROM telepaiement", ())+1).zfill(3)}'
        lid = exe('INSERT INTO telepaiement(reference,contribuable_nom,contribuable_nif,type_impot,montant,montant_paye,statut,mode_paiement,date_echeance,bureau_id,agent_id,notes) VALUES(?,?,?,?,?,?,?,?,?,?,?,?)',
            (ref,d['contribuable_nom'],d.get('contribuable_nif'),d['type_impot'],d['montant'],d.get('montant_paye',0),d.get('statut','En attente'),d.get('mode_paiement'),d.get('date_echeance'),d.get('bureau_id'),u,d.get('notes')))
        return jsonify({'tp_id': lid, 'reference': ref}), 201
    base = 'SELECT tp.*,b.nom bureau_nom,p.nom agent_nom,p.prenom agent_prenom FROM telepaiement tp LEFT JOIN bureau b ON tp.bureau_id=b.bureau_id LEFT JOIN personnel p ON tp.agent_id=p.personnel_id'
    if r == 'chef_centre':
        return jsonify(qry(base + ' ORDER BY tp.date_echeance'))
    elif r == 'chef_bureau':
        bid = qry('SELECT bureau_id FROM personnel WHERE personnel_id=?', (u,), one=True)['bureau_id']
        return jsonify(qry(base + ' WHERE tp.bureau_id=? ORDER BY tp.date_echeance', (bid,)))
    else:
        bid = qry('SELECT bureau_id FROM personnel WHERE personnel_id=?', (u,), one=True)['bureau_id']
        return jsonify(qry(base + ' WHERE tp.bureau_id=? ORDER BY tp.date_echeance', (bid,)))

@app.route('/api/telepaiements/<int:tid>', methods=['PUT'])
def telepaiement_update(tid):
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    d = request.json
    exe('UPDATE telepaiement SET statut=?,montant_paye=?,mode_paiement=?,date_paiement=?,notes=? WHERE tp_id=?',
        (d.get('statut'),d.get('montant_paye'),d.get('mode_paiement'),d.get('date_paiement',str(date.today())),d.get('notes'),tid))
    return jsonify({'ok': True})

@app.route('/api/telepaiements/stats')
def tp_stats():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    return jsonify({
        'par_statut': qry('SELECT statut,COUNT(*) n,ROUND(SUM(montant)/1000000,2) total_m FROM telepaiement GROUP BY statut'),
        'par_bureau': qry('SELECT b.nom,COUNT(*) n,ROUND(SUM(tp.montant)/1000000,2) total_m,ROUND(SUM(tp.montant_paye)/1000000,2) paye_m FROM telepaiement tp JOIN bureau b ON tp.bureau_id=b.bureau_id GROUP BY b.nom'),
        'par_type': qry('SELECT type_impot,COUNT(*) n,ROUND(SUM(montant)/1000000,2) total_m FROM telepaiement GROUP BY type_impot ORDER BY total_m DESC'),
        'total_attendu': qry('SELECT ROUND(SUM(montant)/1000000,2) v FROM telepaiement', one=True)['v'],
        'total_recouvre': qry('SELECT ROUND(SUM(montant_paye)/1000000,2) v FROM telepaiement', one=True)['v'],
    })

# ═══════════════════════════════════════════════════════
#  NOTIFICATIONS, SIGNALEMENTS, PROPOSITIONS, AVIS, CR
# ═══════════════════════════════════════════════════════
@app.route('/api/notifications')
def notifications():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    return jsonify(qry('SELECT n.*,t.libelle tache_libelle FROM notification n LEFT JOIN tache t ON n.tache_id=t.tache_id WHERE n.destinataire_id=? ORDER BY n.dateEnvoie DESC', (uid(),)))

@app.route('/api/notifications/<int:nid>/lire', methods=['PUT'])
def lire_notif(nid):
    exe("UPDATE notification SET statut='Lue' WHERE notification_id=?", (nid,)); return jsonify({'ok': True})

@app.route('/api/notifications/tout-lire', methods=['PUT'])
def tout_lire():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    exe("UPDATE notification SET statut='Lue' WHERE destinataire_id=?", (uid(),)); return jsonify({'ok': True})

@app.route('/api/signalements', methods=['GET', 'POST'])
def signalements():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'POST':
        d = request.json
        sup = qry('SELECT superieur_id FROM personnel WHERE personnel_id=?', (u,), one=True)['superieur_id']
        lid = exe('INSERT INTO signalement(description,dateEnvoie,statut,personnel_id,tache_id,destine_a) VALUES(?,?,?,?,?,?)',
            (d['description'], str(date.today()), 'Ouvert', u, d.get('tache_id'), sup))
        if sup: notif('Nouveau signalement', 'Un collaborateur a émis un signalement. Consultez la section signalements.', 'alerte', sup)
        return jsonify({'signalement_id': lid}), 201
    if r == 'chef_centre':
        return jsonify(qry('SELECT s.*,p.nom auteur,t.libelle tache_lib FROM signalement s LEFT JOIN personnel p ON s.personnel_id=p.personnel_id LEFT JOIN tache t ON s.tache_id=t.tache_id ORDER BY s.dateEnvoie DESC'))
    elif r in ('chef_bureau', 'controleur'):
        return jsonify(qry('SELECT s.*,p.nom auteur,t.libelle tache_lib FROM signalement s LEFT JOIN personnel p ON s.personnel_id=p.personnel_id LEFT JOIN tache t ON s.tache_id=t.tache_id WHERE s.destine_a=? ORDER BY s.dateEnvoie DESC', (u,)))
    else:
        return jsonify(qry('SELECT s.*,t.libelle tache_lib FROM signalement s LEFT JOIN tache t ON s.tache_id=t.tache_id WHERE s.personnel_id=? ORDER BY s.dateEnvoie DESC', (u,)))

@app.route('/api/signalements/<int:sid>/repondre', methods=['PUT'])
def repondre_signal(sid):
    d = request.json
    exe('UPDATE signalement SET reponse=?,statut=? WHERE signalement_id=?', (d.get('reponse'), d.get('statut', 'Traite'), sid))
    sig = qry('SELECT personnel_id FROM signalement WHERE signalement_id=?', (sid,), one=True)
    notif('Signalement traité', 'Votre signalement a reçu une réponse.', 'info', sig['personnel_id'])
    return jsonify({'ok': True})

@app.route('/api/propositions', methods=['GET', 'POST'])
def propositions():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'POST':
        if r not in ('controleur', 'agent'): return jsonify({'error': 'Non autorisé'}), 403
        d = request.json
        sup = qry('SELECT superieur_id FROM personnel WHERE personnel_id=?', (u,), one=True)['superieur_id']
        if not sup: return jsonify({'error': 'Pas de supérieur défini'}), 400
        lid = exe('INSERT INTO proposition_activite(description,type_activite,date_prop,propose_par,destine_a) VALUES(?,?,?,?,?)',
            (d['description'], d.get('type_activite', 'Mission'), str(date.today()), u, sup))
        notif('Nouvelle proposition d\'activité', 'Un collaborateur vous a soumis une proposition d\'activité.', 'proposition', sup)
        return jsonify({'prop_id': lid}), 201
    if r in ('chef_centre', 'chef_bureau'):
        return jsonify(qry('SELECT pa.*,p.nom proposeur,p.prenom proposeur_prenom FROM proposition_activite pa JOIN personnel p ON pa.propose_par=p.personnel_id WHERE pa.destine_a=? ORDER BY pa.date_prop DESC', (u,)))
    else:
        return jsonify(qry('SELECT pa.*,p.nom destinataire FROM proposition_activite pa JOIN personnel p ON pa.destine_a=p.personnel_id WHERE pa.propose_par=? ORDER BY pa.date_prop DESC', (u,)))

@app.route('/api/propositions/<int:pid>/repondre', methods=['PUT'])
def repondre_prop(pid):
    d = request.json
    exe('UPDATE proposition_activite SET statut=?,commentaire_reponse=? WHERE prop_id=?', (d.get('statut'), d.get('commentaire'), pid))
    prop = qry('SELECT propose_par,statut FROM proposition_activite WHERE prop_id=?', (pid,), one=True)
    notif(f'Proposition {d.get("statut","").lower()}', f'Votre proposition d\'activité a été {d.get("statut","").lower()}.', 'info', prop['propose_par'])
    return jsonify({'ok': True})

@app.route('/api/avis', methods=['GET', 'POST'])
def avis():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    if request.method == 'POST':
        d = request.json
        exe('INSERT INTO avis(commentaire,note,dateEnvoie,personnel_id) VALUES(?,?,?,?)', (d.get('commentaire'), d.get('note'), str(date.today()), uid()))
        return jsonify({'ok': True}), 201
    return jsonify(qry('SELECT av.*,p.nom,p.prenom,b.nom bureau_nom FROM avis av LEFT JOIN personnel p ON av.personnel_id=p.personnel_id LEFT JOIN bureau b ON p.bureau_id=b.bureau_id ORDER BY av.dateEnvoie DESC'))

@app.route('/api/comptes-rendus', methods=['GET', 'POST'])
def comptes_rendus():
    if not auth(): return jsonify({'error': 'Non authentifié'}), 401
    u = uid(); r = role()
    if request.method == 'POST':
        if r == 'chef_centre': return jsonify({'error': 'Le chef de centre ne peut pas soumettre de comptes-rendus'}), 403
        d = request.json
        exe('INSERT INTO compteRendu(dateRendue,contenu,statut,personnel_id,tache_id) VALUES(?,?,?,?,?)',
            (str(date.today()), d['contenu'], d.get('statut', 'Soumis'), u, d.get('tache_id')))
        sup = qry('SELECT superieur_id FROM personnel WHERE personnel_id=?', (u,), one=True)['superieur_id']
        if sup: notif('Nouveau compte-rendu', 'Un collaborateur vient de soumettre un compte-rendu.', 'info', sup)
        return jsonify({'ok': True}), 201
    if r == 'chef_centre':
        # Le chef de centre voit TOUS les comptes-rendus
        return jsonify(qry('SELECT cr.*,p.nom,p.prenom,b.nom bureau_nom,t.libelle tache_lib FROM compteRendu cr LEFT JOIN personnel p ON cr.personnel_id=p.personnel_id LEFT JOIN bureau b ON p.bureau_id=b.bureau_id LEFT JOIN tache t ON cr.tache_id=t.tache_id ORDER BY cr.dateRendue DESC'))
    elif r == 'chef_bureau':
        # Le chef de bureau voit les CR de son bureau (contrôleurs + agents)
        bid = qry('SELECT bureau_id FROM personnel WHERE personnel_id=?', (u,), one=True)['bureau_id']
        return jsonify(qry('SELECT cr.*,p.nom,p.prenom,b.nom bureau_nom,t.libelle tache_lib FROM compteRendu cr JOIN personnel p ON cr.personnel_id=p.personnel_id LEFT JOIN bureau b ON p.bureau_id=b.bureau_id LEFT JOIN tache t ON cr.tache_id=t.tache_id WHERE p.bureau_id=? ORDER BY cr.dateRendue DESC', (bid,)))
    elif r == 'controleur':
        # Le contrôleur voit les CR de ses agents directs
        return jsonify(qry('SELECT cr.*,p.nom,p.prenom,t.libelle tache_lib FROM compteRendu cr JOIN personnel p ON cr.personnel_id=p.personnel_id LEFT JOIN tache t ON cr.tache_id=t.tache_id WHERE p.superieur_id=? ORDER BY cr.dateRendue DESC', (u,)))
    else:
        # L'agent voit uniquement ses propres CR
        return jsonify(qry('SELECT cr.*,t.libelle tache_lib FROM compteRendu cr LEFT JOIN tache t ON cr.tache_id=t.tache_id WHERE cr.personnel_id=? ORDER BY cr.dateRendue DESC', (u,)))

# ═══════════════════════════════════════════════════════
#  INTERFACE HTML
# ═══════════════════════════════════════════════════════
HTML = open('/dev/stdin').read() if False else r"""
<!DOCTYPE html><html lang="fr"><head>
<meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>TERRAFISC</title>
<script src="https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js"></script>
<style>
:root{--p:#6B3A2A;--pa:#4e2a1d;--acc:#C9A227;--bg:#fdf8f2;--card:#fff;--txt:#2a1a10;--mut:#8a7060;--bdr:#e2d5c8;--ok:#22a06b;--warn:#e07b28;--err:#d93025;--info:#1967d2;--purple:#6f42c1}
*{box-sizing:border-box;margin:0;padding:0}
body{font-family:'Segoe UI',system-ui,sans-serif;background:var(--bg);color:var(--txt);font-size:14px}
#login-screen{min-height:100vh;display:flex;align-items:center;justify-content:center;background:linear-gradient(145deg,#2a1005 0%,#6B3A2A 60%,#C9A227 100%)}
.lbox{background:#fff;border-radius:18px;padding:44px 38px;width:400px;box-shadow:0 24px 64px rgba(0,0,0,.35)}
.logo{font-size:2.4rem;font-weight:900;letter-spacing:3px;color:var(--p);text-align:center;margin-bottom:4px}.logo span{color:var(--acc)}.logo-tf{display:inline-block;background:var(--p);color:#C9A227;padding:4px 14px;border-radius:8px;font-size:2rem;font-weight:900;letter-spacing:3px;margin-bottom:8px}
.logo-sub{text-align:center;font-size:.8rem;color:var(--mut);margin-bottom:30px;letter-spacing:.5px}
.accounts-hint{background:#f0f4f9;border-radius:10px;padding:14px;margin-bottom:20px;font-size:.75rem}
.accounts-hint b{color:var(--p);display:block;margin-bottom:6px}
.acc-row{display:flex;justify-content:space-between;padding:3px 0;border-bottom:1px solid #e0e7ef;cursor:pointer;color:var(--info)}
.acc-row:last-child{border:none}
#app{display:none}
header{background:var(--p);color:#fff;border-bottom:3px solid var(--acc);height:52px;display:flex;align-items:center;padding:0 20px;gap:14px;position:sticky;top:0;z-index:200;box-shadow:0 2px 8px rgba(0,0,0,.25)}
.brand{font-weight:800;font-size:1.1rem;letter-spacing:1.5px}.brand span{color:var(--acc)}
#role-badge{background:rgba(255,255,255,.12);border-radius:20px;padding:3px 11px;font-size:.7rem;font-weight:700;text-transform:uppercase;letter-spacing:.6px}
.hright{margin-left:auto;display:flex;align-items:center;gap:12px}
#notif-btn{cursor:pointer;background:none;border:none;color:#fff;font-size:1.05rem;position:relative;padding:4px}
#notif-count{position:absolute;top:-5px;right:-5px;background:var(--err);color:#fff;border-radius:50%;width:17px;height:17px;font-size:.62rem;display:none;align-items:center;justify-content:center;font-weight:700}
#logout-btn{background:rgba(255,255,255,.12);border:1px solid rgba(255,255,255,.25);color:#fff;padding:5px 12px;border-radius:7px;cursor:pointer;font-size:.78rem}
.layout{display:flex;min-height:calc(100vh - 52px)}
nav{width:215px;background:#fff;border-right:1px solid var(--bdr);padding:14px 0;flex-shrink:0;position:sticky;top:52px;height:calc(100vh - 52px);overflow-y:auto}
.nsec{padding:10px 16px 4px;font-size:.65rem;font-weight:700;text-transform:uppercase;letter-spacing:.9px;color:var(--mut)}
.ni{display:flex;align-items:center;gap:9px;padding:9px 18px;cursor:pointer;color:var(--txt);font-size:.84rem;border-left:3px solid transparent;transition:.12s}
.ni:hover{background:#f0f4f9;color:var(--p)}
.ni.active{background:#fdf3e0;color:var(--p);border-left-color:var(--acc);font-weight:600}
.nico{font-size:14px;width:18px;text-align:center}
.main{flex:1;padding:22px;overflow-y:auto}
.page{display:none}.page.active{display:block}
.ptitle{font-size:1.2rem;font-weight:700;color:var(--p);margin-bottom:18px;display:flex;align-items:center;gap:10px;flex-wrap:wrap}
.gstats{display:grid;grid-template-columns:repeat(auto-fill,minmax(150px,1fr));gap:12px;margin-bottom:20px}
.stat{background:#fff;border-radius:10px;padding:16px;border:1px solid var(--bdr);border-top:3px solid var(--p)}
.stat .n{font-size:1.9rem;font-weight:800;color:var(--p);line-height:1}.stat .l{font-size:.73rem;color:var(--mut);margin-top:3px}
.stat.acc{border-top-color:var(--acc)}.stat.acc .n{color:var(--acc)}
.stat.ok{border-top-color:var(--ok)}.stat.ok .n{color:var(--ok)}
.stat.warn{border-top-color:var(--warn)}.stat.warn .n{color:var(--warn)}
.stat.err{border-top-color:var(--err)}.stat.err .n{color:var(--err)}
.stat.purple{border-top-color:var(--purple)}.stat.purple .n{color:var(--purple)}
.card{background:#fff;border-radius:11px;border:1px solid var(--bdr);margin-bottom:18px;overflow:hidden}
.chead{padding:13px 18px;border-bottom:1px solid var(--bdr);display:flex;align-items:center;justify-content:space-between;background:#fafbfc}
.chead h3{font-size:.9rem;font-weight:700;color:var(--p)}
.cbody{padding:16px 18px}
table{width:100%;border-collapse:collapse}
th{background:#f4f7fb;color:var(--mut);font-size:.72rem;font-weight:700;text-transform:uppercase;letter-spacing:.4px;padding:10px 12px;text-align:left;border-bottom:2px solid var(--bdr)}
td{padding:10px 12px;border-bottom:1px solid #f0f3f8;font-size:.83rem;vertical-align:middle}
tr:last-child td{border:none}tr:hover td{background:#f8faff}
.btn{padding:7px 13px;border:none;border-radius:7px;cursor:pointer;font-size:.8rem;font-weight:600;transition:.13s;display:inline-flex;align-items:center;gap:5px;white-space:nowrap}
.bp{background:var(--p);color:#fff}.bp:hover{background:var(--pa)}
.ba{background:var(--acc);color:#fff}.ba:hover{background:#c88018}
.bd{background:var(--err);color:#fff;padding:4px 8px;font-size:.73rem}
.bsuc{background:var(--ok);color:#fff}
.bo{background:transparent;border:1px solid var(--bdr);color:var(--txt)}.bo:hover{background:#f0f4f9}
.bsm{padding:4px 9px;font-size:.73rem}
.badge{display:inline-flex;align-items:center;padding:3px 8px;border-radius:20px;font-size:.7rem;font-weight:700;white-space:nowrap}
.bs{background:#d2f5e8;color:#0d6b42}.bw{background:#fff3cd;color:#7a5200}.be{background:#fce8e6;color:#9c1f1f}.bi{background:#d3e5fb;color:#1254a0}.bm{background:#eee;color:#555}.bp2{background:#ede7f6;color:#4a2faa}
.fg{margin-bottom:13px}.fg label{display:block;font-size:.78rem;font-weight:700;color:var(--mut);margin-bottom:4px}
input,select,textarea{width:100%;padding:8px 10px;border:1.5px solid var(--bdr);border-radius:7px;font-size:.86rem;color:var(--txt);background:#fff;outline:none;transition:border .13s}
input:focus,select:focus,textarea:focus{border-color:var(--p)}
.fr{display:grid;grid-template-columns:1fr 1fr;gap:12px}
.mo{display:none;position:fixed;inset:0;background:rgba(0,0,0,.55);z-index:1000;overflow-y:auto;padding:28px 14px;align-items:flex-start;justify-content:center}
.mo.open{display:flex}
.mobox{background:#fff;border-radius:14px;padding:26px;width:100%;max-width:520px;margin:auto}
.mobox h3{font-size:1rem;font-weight:700;color:var(--p);margin-bottom:18px}
.moftr{display:flex;gap:10px;justify-content:flex-end;margin-top:18px}
.np{position:fixed;top:52px;right:0;width:340px;background:#fff;border-left:1px solid var(--bdr);height:calc(100vh - 52px);overflow-y:auto;z-index:300;transform:translateX(100%);transition:.22s}
.np.open{transform:none}
.nphead{padding:13px 16px;border-bottom:1px solid var(--bdr);display:flex;align-items:center;justify-content:space-between;background:#f5f8fc}
.nitem{padding:11px 14px;border-bottom:1px solid #f0f3f8;cursor:pointer;transition:.1s}
.nitem:hover{background:#f5f8fc}
.nitem.unread{background:#e8f0fd;border-left:3px solid var(--p)}
.ntit{font-weight:700;font-size:.8rem;margin-bottom:2px}.ndesc{font-size:.76rem;color:var(--mut);line-height:1.4}.ntime{font-size:.68rem;color:var(--mut);margin-top:3px}
.em-card{background:linear-gradient(135deg,#1a3a5c,#2d6a9f);color:#fff;border-radius:14px;padding:20px;margin-bottom:16px;position:relative;overflow:hidden}
.em-card::after{content:'🏆';position:absolute;right:16px;top:50%;transform:translateY(-50%);font-size:3rem;opacity:.3}
.em-card .em-month{font-size:.75rem;opacity:.8;text-transform:uppercase;letter-spacing:.8px}
.em-card .em-name{font-size:1.3rem;font-weight:800;margin:4px 0}
.em-card .em-bureau{font-size:.8rem;opacity:.85}
.em-card .em-note{font-size:.85rem;margin-top:8px;opacity:.9}
.tp-statut-paye{color:#0d6b42;font-weight:700}.tp-statut-retard{color:#9c1f1f;font-weight:700}.tp-statut-partiel{color:#7a5200;font-weight:700}
.pgbar{background:#e8edf5;border-radius:6px;height:7px;overflow:hidden;margin-top:4px}
.pgfill{height:100%;border-radius:6px;background:var(--p);transition:width .3s}
.av{width:32px;height:32px;border-radius:50%;display:flex;align-items:center;justify-content:center;font-size:.72rem;font-weight:800;flex-shrink:0}
.role-chip{display:inline-flex;padding:2px 8px;border-radius:20px;font-size:.67rem;font-weight:800;text-transform:uppercase;letter-spacing:.5px}
.rc-cc{background:#6B3A2A;color:#C9A227}.rc-cb{background:#4e2a1d;color:#fff}.rc-ctrl{background:#8a6020;color:#fff}.rc-agent{background:#5a4030;color:#fff}
.acts{display:flex;gap:5px;flex-wrap:wrap}
.cht{position:relative;height:190px}
.grids2{display:grid;grid-template-columns:1fr 1fr;gap:16px}
.grids3{display:grid;grid-template-columns:1fr 1fr 1fr;gap:16px}
#toast{position:fixed;bottom:22px;right:22px;color:#fff;padding:11px 16px;border-radius:9px;display:none;z-index:9999;font-size:.84rem;max-width:310px;box-shadow:0 4px 18px rgba(0,0,0,.3)}
.empty{text-align:center;padding:38px 20px;color:var(--mut)}
.empty .eico{font-size:2.2rem;margin-bottom:8px}
.prio-H td:first-child{border-left:3px solid var(--err)}
.prio-U td:first-child{border-left:3px solid var(--purple)}
@media(max-width:700px){nav{display:none}.grids2,.grids3{grid-template-columns:1fr}.fr{grid-template-columns:1fr}}
</style></head><body>

<div id="login-screen">
 <div class="lbox">
  <div style="text-align:center;margin-bottom:8px"><div class="logo-tf">TF</div></div>
  <div class="logo">TERRA<span>FISC</span></div>
  <div class="logo-sub">Système de Gestion Administrative & Fiscale</div>

  <div id="lerr" style="background:#fce8e6;color:#9c1f1f;padding:8px 12px;border-radius:7px;font-size:.8rem;margin-bottom:12px;display:none"></div>
  <div class="fg"><label>Email</label><input id="l-e" type="email"></div>
  <div class="fg"><label>Mot de passe</label><input id="l-p" type="password" value="admin123"></div>
  <button class="btn bp" style="width:100%;padding:10px;margin-top:4px;font-size:.9rem" onclick="doLogin()">Se connecter</button>
 </div>
</div>

<div id="app">
 <header>
  <div class="brand"><span style="background:var(--acc);color:var(--p);padding:1px 6px;border-radius:4px;font-size:.8rem;margin-right:4px">TF</span>TERRA<span>FISC</span></div>
  <div id="role-badge"></div>
  <div class="hright">
   <span id="uname" style="font-size:.84rem;opacity:.9"></span>
   <button id="notif-btn" onclick="toggleNP()" title="Notifications">🔔<span id="notif-count">0</span></button>
   <button id="logout-btn" onclick="doLogout()">Déconnexion</button>
  </div>
 </header>
 <div class="layout">
  <nav id="nav"></nav>
  <div class="main">
   <div class="page active" id="pg-dashboard"></div>
   <div class="page" id="pg-bureaux"></div>
   <div class="page" id="pg-activites"></div>
   <div class="page" id="pg-taches"></div>
   <div class="page" id="pg-affecter"></div>
   <div class="page" id="pg-personnel"></div>
   <div class="page" id="pg-performances"></div>
   <div class="page" id="pg-employe-mois"></div>
   <div class="page" id="pg-telepaiement"></div>
   <div class="page" id="pg-signalements"></div>
   <div class="page" id="pg-propositions"></div>
   <div class="page" id="pg-cr"></div>
   <div class="page" id="pg-avis"></div>
   <div class="page" id="pg-profil"></div>
  </div>
 </div>
 <div class="np" id="np">
  <div class="nphead"><b style="font-size:.88rem">🔔 Notifications</b><button class="btn bo bsm" onclick="toutLire()">Tout lire</button></div>
  <div id="nlist"></div>
 </div>
</div>

<!-- MODALS -->
<div class="mo" id="mo-activite"><div class="mobox">
 <h3>Nouvelle activité</h3>
 <div class="fr"><div class="fg"><label>Nom</label><input id="ma-n"></div><div class="fg"><label>Type</label><select id="ma-t"><option>Mission</option><option>Atelier</option><option>Séminaire</option><option>Forum</option><option>Colloque</option><option>Audit</option><option>Formation</option><option>Projet</option></select></div></div>
 <div class="fr"><div class="fg"><label>Début</label><input type="date" id="ma-d"></div><div class="fg"><label>Fin</label><input type="date" id="ma-f"></div></div>
 <div class="fg"><label>Bureau</label><select id="ma-b"></select></div>
 <div class="fg"><label>Description</label><textarea id="ma-desc" rows="3"></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-activite')">Annuler</button><button class="btn bp" onclick="saveActivite()">Enregistrer</button></div>
</div></div>

<div class="mo" id="mo-tache"><div class="mobox">
 <h3>Nouvelle tâche</h3>
 <div class="fr"><div class="fg"><label>Libellé</label><input id="mt-l"></div><div class="fg"><label>Activité</label><select id="mt-a"></select></div></div>
 <div class="fr"><div class="fg"><label>Début</label><input type="date" id="mt-d"></div><div class="fg"><label>Fin</label><input type="date" id="mt-f"></div></div>
 <div class="fr"><div class="fg"><label>Priorité</label><select id="mt-p"><option>Normale</option><option>Haute</option><option>Urgente</option></select></div><div class="fg"><label>Statut initial</label><select id="mt-s"><option>Non demarre</option><option>En cours</option></select></div></div>
 <div class="fg"><label>Description</label><textarea id="mt-desc" rows="3"></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-tache')">Annuler</button><button class="btn bp" onclick="saveTache()">Enregistrer</button></div>
</div></div>

<div class="mo" id="mo-affect"><div class="mobox">
 <h3>Affecter un subordonné</h3>
 <input type="hidden" id="aff-tid">
 <div class="fg"><label>Tâche</label><input id="aff-tl" readonly style="background:#f5f8fc"></div>
 <div class="fg"><label>Subordonné direct</label><select id="aff-pid"></select></div>
 <div class="fg"><label>Rôle dans la tâche</label><input id="aff-r" value="Executant"></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-affect')">Annuler</button><button class="btn bp" onclick="saveAffect()">✅ Affecter & Notifier</button></div>
</div></div>

<div class="mo" id="mo-noter"><div class="mobox">
 <h3>Évaluation de performance</h3>
 <input type="hidden" id="n-pid">
 <div class="fg"><label>Collaborateur évalué</label><input id="n-nom" readonly style="background:#f5f8fc"></div>
 <div class="fr"><div class="fg"><label>Efficacité (%)</label><input type="number" id="n-eff" min="0" max="100"></div><div class="fg"><label>Note /20</label><input type="number" id="n-note" min="0" max="20"></div></div>
 <div class="fg"><label>Prime (optionnel)</label><input id="n-prime" placeholder="ex: 75 000 FCFA"></div>
 <div class="fg"><label>Commentaire</label><textarea id="n-comm" rows="3"></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-noter')">Annuler</button><button class="btn bsuc" onclick="saveNote()">Enregistrer</button></div>
</div></div>

<div class="mo" id="mo-em"><div class="mobox">
 <h3>🏆 Désigner l'Employé du mois</h3>
 <div class="fr"><div class="fg"><label>Mois</label><select id="em-mois"><option value="1">Janvier</option><option value="2">Février</option><option value="3">Mars</option><option value="4">Avril</option><option value="5">Mai</option><option value="6">Juin</option><option value="7">Juillet</option><option value="8">Août</option><option value="9">Septembre</option><option value="10">Octobre</option><option value="11">Novembre</option><option value="12">Décembre</option></select></div><div class="fg"><label>Année</label><input type="number" id="em-annee" value="2025"></div></div>
 <div class="fg"><label>Agent désigné</label><select id="em-pid"></select></div>
 <div class="fg"><label>Note finale /20</label><input type="number" id="em-note" min="0" max="20" step="0.1"></div>
 <div class="fg"><label>Motif de la désignation</label><textarea id="em-motif" rows="3" placeholder="Décrivez les mérites de cet agent..."></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-em')">Annuler</button><button class="btn ba" onclick="saveEM()">🏆 Désigner</button></div>
</div></div>

<div class="mo" id="mo-tp"><div class="mobox" style="max-width:560px">
 <h3>Nouveau télépaiement</h3>
 <div class="fr"><div class="fg"><label>Contribuable</label><input id="tp-cn"></div><div class="fg"><label>NIF / NINEA</label><input id="tp-nif" placeholder="SN000000"></div></div>
 <div class="fr"><div class="fg"><label>Type d'impôt / taxe</label><select id="tp-type"><option>Impôt sur les Sociétés</option><option>TVA mensuelle</option><option>Taxe foncière</option><option>Acompte IS</option><option>Taxe sur les salaires</option><option>Droits d'enregistrement</option><option>Taxe de plus-value immobilière</option><option>Droits de bornage</option><option>Contribution foncière bâtie</option><option>Autre</option></select></div><div class="fg"><label>Bureau</label><select id="tp-b"></select></div></div>
 <div class="fr"><div class="fg"><label>Montant dû (FCFA)</label><input type="number" id="tp-mt"></div><div class="fg"><label>Échéance</label><input type="date" id="tp-ech"></div></div>
 <div class="fg"><label>Notes</label><textarea id="tp-notes" rows="2"></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-tp')">Annuler</button><button class="btn bp" onclick="saveTP()">Enregistrer</button></div>
</div></div>

<div class="mo" id="mo-tp-update"><div class="mobox">
 <h3>Mettre à jour le paiement</h3>
 <input type="hidden" id="tpu-id">
 <div class="fg"><label>Statut</label><select id="tpu-statut"><option>En attente</option><option>Partiel</option><option>Paye</option><option>En retard</option></select></div>
 <div class="fr"><div class="fg"><label>Montant reçu (FCFA)</label><input type="number" id="tpu-mt"></div><div class="fg"><label>Mode de paiement</label><select id="tpu-mode"><option>Virement bancaire</option><option>Mobile Money (Wave)</option><option>Mobile Money (Orange)</option><option>Carte bancaire</option><option>Espèces (guichet)</option><option>Chèque</option></select></div></div>
 <div class="fg"><label>Notes</label><textarea id="tpu-notes" rows="2"></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-tp-update')">Annuler</button><button class="btn bsuc" onclick="saveTPUpdate()">Confirmer paiement</button></div>
</div></div>

<div class="mo" id="mo-personnel"><div class="mobox">
 <h3>Ajouter un membre du personnel</h3>
 <div class="fg" style="text-align:center">
  <label>Photo de profil (optionnel)</label>
  <div id="mp-photo-preview" style="width:80px;height:80px;border-radius:50%;background:#f0e8e0;border:2px dashed var(--bdr);margin:6px auto;display:flex;align-items:center;justify-content:center;font-size:1.8rem;overflow:hidden"><span id="mp-photo-icon">📷</span><img id="mp-photo-img" style="display:none;width:100%;height:100%;object-fit:cover"></div>
  <input type="file" id="mp-photo-file" accept="image/*" style="display:none" onchange="previewPhoto(this,'mp-photo-img','mp-photo-icon','mp-photo-b64')">
  <button class="btn bo bsm" onclick="document.getElementById('mp-photo-file').click()">Choisir une photo</button>
  <input type="hidden" id="mp-photo-b64">
 </div>
 <p style="font-size:.72rem;color:var(--mut);text-align:center;margin:-6px 0 8px">📧 L'email sera généré automatiquement à partir du nom et prénom</p>
 <div class="fr"><div class="fg"><label>Nom</label><input id="mp-n"></div><div class="fg"><label>Prénom</label><input id="mp-pr"></div></div>
 <div class="fr"><div class="fg"><label>Téléphone</label><input id="mp-t"></div><div class="fg"><label>Date de naissance</label><input type="date" id="mp-dob"></div></div>
 <div class="fr"><div class="fg"><label>Rôle</label><select id="mp-r"><option value="chef_bureau">Chef de Bureau</option><option value="controleur">Contrôleur</option><option value="agent">Agent</option></select></div><div class="fg"><label>Bureau</label><select id="mp-b"></select></div></div>
 <div class="fr"><div class="fg"><label>Supérieur N+1</label><select id="mp-s"><option value="">-- Aucun --</option></select></div><div class="fg"><label>Fonction</label><input id="mp-f"></div></div>

 <div class="moftr"><button class="btn bo" onclick="cm('mo-personnel')">Annuler</button><button class="btn bp" onclick="savePersonnel()">Enregistrer</button></div>
</div></div>

<div class="mo" id="mo-prop"><div class="mobox">
 <h3>Proposer une activité à mon N+1</h3>
 <div class="fg"><label>Type d'activité</label><select id="pp-t"><option>Mission</option><option>Atelier</option><option>Séminaire</option><option>Forum</option><option>Formation</option></select></div>
 <div class="fg"><label>Description détaillée</label><textarea id="pp-d" rows="4" placeholder="Décrivez l'activité que vous proposez et son intérêt..."></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-prop')">Annuler</button><button class="btn bp" onclick="saveProp()">Envoyer à mon N+1</button></div>
</div></div>

<div class="mo" id="mo-signal"><div class="mobox">
 <h3>Faire un signalement</h3>
 <div class="fg"><label>Tâche concernée</label><select id="sg-t"><option value="">-- Aucune --</option></select></div>
 <div class="fg"><label>Description du problème</label><textarea id="sg-d" rows="4"></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-signal')">Annuler</button><button class="btn ba" onclick="saveSignal()">Envoyer au N+1</button></div>
</div></div>

<div class="mo" id="mo-cr"><div class="mobox">
 <h3>Soumettre un compte-rendu</h3>
 <div class="fg"><label>Tâche concernée</label><select id="cr-t"><option value="">-- Général --</option></select></div>
 <div class="fg"><label>Contenu</label><textarea id="cr-c" rows="6"></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-cr')">Annuler</button><button class="btn bp" onclick="saveCR()">Soumettre</button></div>
</div></div>

<div class="mo" id="mo-avis"><div class="mobox">
 <h3>Donner un avis sur l'application</h3>
 <div class="fg"><label>Note globale</label><select id="av-n"><option value="5">⭐⭐⭐⭐⭐ Excellent</option><option value="4">⭐⭐⭐⭐ Bien</option><option value="3">⭐⭐⭐ Moyen</option><option value="2">⭐⭐ Insuffisant</option><option value="1">⭐ Mauvais</option></select></div>
 <div class="fg"><label>Commentaire</label><textarea id="av-c" rows="3"></textarea></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-avis')">Annuler</button><button class="btn ba" onclick="saveAvis()">Soumettre</button></div>
</div></div>

<div class="mo" id="mo-gantt"><div class="mobox" style="max-width:700px">
 <h3 id="gantt-t">Diagramme de Gantt</h3>
 <div id="gantt-c" style="margin-top:14px"></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-gantt')">Fermer</button></div>
</div></div>

<div class="mo" id="mo-hist"><div class="mobox">
 <h3>Historique des affectations</h3>
 <div id="hist-c" style="margin-top:12px"></div>
 <div class="moftr"><button class="btn bo" onclick="cm('mo-hist')">Fermer</button></div>
</div></div>

<div class="mo" id="mo-gerer-aff"><div class="mobox" style="max-width:580px">
 <h3 id="ga-titre">👥 Gérer les affectations</h3>
 <input type="hidden" id="ga-tid">
 <div id="ga-liste" style="margin:14px 0"></div>
 <div class="moftr">
  <button class="btn bo" onclick="cm('mo-gerer-aff')">Fermer</button>
  <button class="btn bp" onclick="openAff(document.getElementById('ga-tid').value, document.getElementById('ga-titre').textContent.replace('👥 Gérer — ',''))">+ Affecter un agent</button>
 </div>
</div></div>

<div id="toast"></div>

<script>
let CU=null, charts={};
const RFR={chef_centre:'Chef de Centre',chef_bureau:'Chef de Bureau',controleur:'Contrôleur',agent:'Agent'};
const RCOL={chef_centre:'#1a3a5c',chef_bureau:'#145c3a',controleur:'#7a4a08',agent:'#3d3d3d'};
const MFR=['','Janvier','Février','Mars','Avril','Mai','Juin','Juillet','Août','Septembre','Octobre','Novembre','Décembre'];

function api(u,o={}){return fetch(u,{credentials:'include',headers:{'Content-Type':'application/json'},...o}).then(r=>r.json())}
function post(u,d){return api(u,{method:'POST',body:JSON.stringify(d)})}
function put(u,d){return api(u,{method:'PUT',body:JSON.stringify(d)})}
function del(u){return api(u,{method:'DELETE'})}
function om(id){document.getElementById(id).classList.add('open')}
function cm(id){document.getElementById(id).classList.remove('open')}
document.querySelectorAll('.mo').forEach(m=>m.addEventListener('click',e=>{if(e.target===m)m.classList.remove('open')}))

function toast(msg,type='info'){
 const t=document.getElementById('toast');
 t.textContent=msg;t.style.display='block';
 t.style.background=type==='err'?'#9c1f1f':type==='ok'?'#0d6b42':type==='warn'?'#7a5200':'#1a3a5c';
 try{
  const ac=new AudioContext(),osc=ac.createOscillator(),g=ac.createGain();
  osc.connect(g);g.connect(ac.destination);
  osc.frequency.setValueAtTime(type==='err'?220:680,ac.currentTime);
  osc.frequency.exponentialRampToValueAtTime(type==='err'?180:880,ac.currentTime+0.08);
  g.gain.setValueAtTime(0.12,ac.currentTime);g.gain.exponentialRampToValueAtTime(0.001,ac.currentTime+0.25);
  osc.start();osc.stop(ac.currentTime+0.25);
 }catch(e){}
 if(navigator.vibrate)navigator.vibrate(type==='err'?[80,40,80]:[50]);
 setTimeout(()=>t.style.display='none',3200);
}

function badge(s){
 const m={Terminee:'bs',Paye:'bs','En cours':'bw',Partiel:'bw',Planifiee:'bi','Non demarre':'bm',
  'Non lue':'bw',Lue:'bm',Ouvert:'be',Traite:'bs',Acceptee:'bs',Refusee:'be','En attente':'bw',
  Soumise:'bi',Soumis:'bw',Valide:'bs',Brouillon:'bm',Haute:'be',Normale:'bi',Urgente:'bp2',
  'En retard':'be'};
 return `<span class="badge ${m[s]||'bm'}">${s||'—'}</span>`;
}
function stars(n){return'<span style="color:#e8a020">'+('★'.repeat(n)+'☆'.repeat(5-n))+'</span>';}
function fmt(n){return n!=null?Number(n).toLocaleString('fr-FR'):'—';}
function pct(a,b){return b?Math.round(a/b*100):0;}
function av(nom,prenom,role){const i=((nom||'?')[0]+(prenom||'?')[0]).toUpperCase();return `<div class="av" style="background:${RCOL[role]||'#888'};color:#fff">${i}</div>`;}

function showPage(name){
 document.querySelectorAll('.page').forEach(p=>p.classList.remove('active'));
 document.querySelectorAll('.ni').forEach(n=>n.classList.remove('active'));
 document.getElementById('pg-'+name).classList.add('active');
 document.querySelector(`[data-p="${name}"]`)?.classList.add('active');
 LOADS[name]?.();
}

function setLogin(email){document.getElementById('l-e').value=email;}

async function doLogin(){
 const r=await post('/api/login',{email:document.getElementById('l-e').value,password:document.getElementById('l-p').value});
 if(r.error){const e=document.getElementById('lerr');e.textContent=r.error;e.style.display='block';return;}
 CU=r;initApp();
}

async function doLogout(){
 await post('/api/logout',{});CU=null;
 document.getElementById('app').style.display='none';
 document.getElementById('login-screen').style.display='flex';
}

function initApp(){
 document.getElementById('login-screen').style.display='none';
 document.getElementById('app').style.display='block';
 document.getElementById('role-badge').textContent=RFR[CU.role]||CU.role;
 document.getElementById('uname').textContent=`${CU.prenom||''} ${CU.nom}`;
 buildNav();loadNotifs();populateSels();
 showPage('dashboard');
 setInterval(loadNotifs,25000);
}

function buildNav(){
 const r=CU.role;
 const items=[
  {s:'Tableau de bord'},{p:'dashboard',i:'⊞',l:'Vue générale'},
  {s:'Activités & Tâches'},
  {p:'activites',i:'📋',l:'Activités'},{p:'taches',i:'✅',l:'Tâches'},
  ...['chef_centre','chef_bureau','controleur'].includes(r)?[{p:'affecter',i:'👤',l:'Affectations'}]:[],
  {s:'Mon équipe'},
  ...r!=='agent'?[{p:'personnel',i:'👥',l:'Personnel'}]:[],
  {p:'performances',i:'📊',l:r==='agent'?'Mes évaluations':'Performances'},
  {p:'employe-mois',i:'🏆',l:'Employé du mois'},
  {s:'Fiscal & Paiement'},
  {p:'telepaiement',i:'💳',l:'Télépaiements'},
  {s:'Mon profil'},
  {p:'profil',i:'🪪',l:'Informations personnelles'},
  {s:'Communication'},
  {p:'signalements',i:'⚠️',l:'Signalements'},
  {p:'propositions',i:'💡',l:['chef_bureau','chef_centre'].includes(r)?'Propositions reçues':'Proposer activité'},
  ...r!=='chef_centre'?[{p:'cr',i:'📄',l:'Comptes-rendus'}]:[],
  {p:'avis',i:'⭐',l:'Avis application'},
  ...r==='chef_centre'?[{s:'Administration'},{p:'bureaux',i:'🏢',l:'Bureaux'}]:[],
 ];
 document.getElementById('nav').innerHTML=items.map(i=>
  i.s?`<div class="nsec">${i.s}</div>`:
  `<div class="ni" data-p="${i.p}" onclick="showPage('${i.p}')"><span class="nico">${i.i}</span>${i.l}</div>`
 ).join('');
}

// ── NOTIFICATIONS ──
async function loadNotifs(){
 const rows=await api('/api/notifications');if(rows.error)return;
 const unread=rows.filter(r=>r.statut==='Non lue');
 const cnt=document.getElementById('notif-count');
 cnt.textContent=unread.length;cnt.style.display=unread.length?'flex':'none';
 if(window._nc!==undefined&&unread.length>window._nc&&unread.length>0)toast(`🔔 ${unread.length} notification(s) non lue(s)`);
 window._nc=unread.length;
 const tico={tache:'✅',performance:'📊',employe_mois:'🏆',proposition:'💡',alerte:'⚠️',info:'ℹ️'};
 document.getElementById('nlist').innerHTML=rows.length===0
  ?'<div class="empty"><div class="eico">🔔</div><p>Aucune notification</p></div>'
  :rows.map(r=>`<div class="nitem ${r.statut==='Non lue'?'unread':''}" onclick="lireNotif(${r.notification_id},this)">
   <div class="ntit">${tico[r.type_notif]||'•'} ${r.titre}</div>
   <div class="ndesc">${r.description}</div>
   <div class="ntime">${(r.dateEnvoie||'').substring(0,16)}</div></div>`).join('');
}
async function lireNotif(id,el){await put('/api/notifications/'+id+'/lire',{});el.classList.remove('unread');loadNotifs();}
async function toutLire(){await put('/api/notifications/tout-lire',{});loadNotifs();toast('Toutes les notifications lues','ok');}
function toggleNP(){document.getElementById('np').classList.toggle('open');}

function mkChart(id,data,type){
 if(charts[id])charts[id].destroy();
 const el=document.getElementById(id);if(!el)return;
 const lk=Object.keys(data[0]||{}).find(k=>!['n','total','total_m','paye_m'].includes(k));
 const vk=data[0]&&('total_m' in data[0])?'total_m':'n';
 const C=['#1a3a5c','#e8a020','#22a06b','#d93025','#6f42c1','#1967d2','#e07b28'];
 charts[id]=new Chart(el,{type,data:{labels:data.map(r=>r[lk]),datasets:[{data:data.map(r=>r[vk]),backgroundColor:C,borderColor:type==='bar'?C:'#fff',borderWidth:2,borderRadius:type==='bar'?4:0}]},
  options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{position:type==='doughnut'?'right':'bottom',labels:{font:{size:11}}}}}});
}

// ── DASHBOARD ──
async function loadDashboard(){
 const d=await api('/api/dashboard');if(d.error)return;
 const r=CU.role;const el=document.getElementById('pg-dashboard');
 if(r==='chef_centre'){
  const tpTotal=d.tp_stats?.reduce((s,r)=>s+r.total_m,0).toFixed(2)||0;
  const tpPaye=d.tp_stats?.find(r=>r.statut==='Paye')?.total_m||0;
  el.innerHTML=`<div class="ptitle">⊞ Vue d'ensemble — Chef de Centre</div>
   <div class="gstats">
    <div class="stat"><div class="n">${d.nb_bureaux}</div><div class="l">Bureaux</div></div>
    <div class="stat acc"><div class="n">${d.nb_personnel}</div><div class="l">Personnel total</div></div>
    <div class="stat"><div class="n">${d.nb_activites}</div><div class="l">Activités</div></div>
    <div class="stat warn"><div class="n">${d.nb_taches}</div><div class="l">Tâches</div></div>
    <div class="stat ok"><div class="n">${d.nb_notifs}</div><div class="l">Notifs non lues</div></div>
    <div class="stat purple"><div class="n">${tpPaye}M</div><div class="l">FCFA recouvrés</div></div>
   </div>
   ${d.employe_mois?.length?`<div class="grids3" style="margin-bottom:18px">${d.employe_mois.map(em=>`<div class="em-card"><div class="em-month">${MFR[em.mois]} ${em.annee}</div><div class="em-name">${em.nom} ${em.prenom||''}</div><div class="em-bureau">${em.bureau_nom}</div><div class="em-note">Note : ${em.note_finale}/20</div></div>`).join('')}</div>`:''}
   <div class="grids3">
    <div class="card"><div class="chead"><h3>Activités / statut</h3></div><div class="cbody"><div class="cht"><canvas id="ch-a"></canvas></div></div></div>
    <div class="card"><div class="chead"><h3>Tâches / statut</h3></div><div class="cbody"><div class="cht"><canvas id="ch-t"></canvas></div></div></div>
    <div class="card"><div class="chead"><h3>Activités / bureau</h3></div><div class="cbody"><div class="cht"><canvas id="ch-b"></canvas></div></div></div>
   </div>
   <div class="card" style="margin-top:16px"><div class="chead"><h3>🏅 Dernières performances</h3></div><div class="cbody">
    <table><thead><tr><th>Agent</th><th>Bureau</th><th>Note</th><th>Efficacité</th></tr></thead><tbody>
    ${(d.top_performances||[]).map(p=>`<tr><td><b>${p.nom} ${p.prenom||''}</b></td><td>${p.bureau_nom}</td><td><b style="font-size:1rem;color:${p.note>=15?'#22a06b':p.note>=10?'#e07b28':'#d93025'}">${p.note}</b>/20</td><td><div class="pgbar"><div class="pgfill" style="width:${p.efficacite}%"></div></div></td></tr>`).join('')}
    </tbody></table></div></div>`;
  mkChart('ch-a',d.activites_statut,'doughnut');
  mkChart('ch-t',d.taches_statut,'doughnut');
  mkChart('ch-b',d.bureaux_activites,'bar');
 }
 else if(r==='chef_bureau'){
  el.innerHTML=`<div class="ptitle">⊞ Mon Bureau — Chef de Bureau</div>
   <div class="gstats">
    <div class="stat"><div class="n">${d.nb_controleurs}</div><div class="l">Contrôleurs</div></div>
    <div class="stat acc"><div class="n">${d.nb_agents}</div><div class="l">Agents</div></div>
    <div class="stat"><div class="n">${d.nb_activites}</div><div class="l">Activités</div></div>
    <div class="stat warn"><div class="n">${d.nb_taches}</div><div class="l">Tâches</div></div>
    <div class="stat ok"><div class="n">${d.nb_notifs}</div><div class="l">Notifs non lues</div></div>
    <div class="stat err"><div class="n">${d.signalements_ouverts}</div><div class="l">Signalements</div></div>
    <div class="stat purple"><div class="n">${d.propositions_recues}</div><div class="l">Propositions</div></div>
   </div>
   ${d.employe_mois?.length?`<div style="margin-bottom:18px">${d.employe_mois.map(em=>`<div class="em-card" style="max-width:360px"><div class="em-month">Employé du mois — ${MFR[em.mois]} ${em.annee}</div><div class="em-name">${em.nom} ${em.prenom||''}</div><div class="em-note">${em.motif||''}</div></div>`).join('')}</div>`:''}
   <div class="grids2">
    <div class="card"><div class="chead"><h3>Tâches du bureau</h3></div><div class="cbody"><div class="cht"><canvas id="ch-tb"></canvas></div></div></div>
    <div class="card"><div class="chead"><h3>Télépaiements du bureau</h3></div><div class="cbody"><div class="cht"><canvas id="ch-tpb"></canvas></div></div></div>
   </div>`;
  mkChart('ch-tb',d.taches_statut,'doughnut');
  if(d.tp_bureau?.length) mkChart('ch-tpb',d.tp_bureau,'doughnut');
 }
 else if(r==='controleur'){
  el.innerHTML=`<div class="ptitle">⊞ Mon équipe — Contrôleur</div>
   <div class="gstats">
    <div class="stat"><div class="n">${d.nb_agents}</div><div class="l">Agents supervisés</div></div>
    <div class="stat acc"><div class="n">${d.nb_taches_assignees}</div><div class="l">Tâches assignées</div></div>
    <div class="stat warn"><div class="n">${d.nb_notifs}</div><div class="l">Notifs non lues</div></div>
    <div class="stat err"><div class="n">${d.signalements_recus}</div><div class="l">Signalements reçus</div></div>
   </div>
   <div class="grids2">
    <div class="card"><div class="chead"><h3>Avancement des tâches</h3></div><div class="cbody"><div class="cht"><canvas id="ch-tc"></canvas></div></div></div>
    <div class="card"><div class="chead"><h3>Mes tâches</h3><button class="btn bo bsm" onclick="showPage('taches')">Tout voir</button></div><div class="cbody">
    ${(d.mes_taches||[]).slice(0,6).map(t=>`<div style="display:flex;justify-content:space-between;align-items:center;padding:7px 0;border-bottom:1px solid #f0f3f8">
     <div><div style="font-size:.82rem;font-weight:600">${t.libelle}</div><div style="font-size:.72rem;color:var(--mut)">${t.act_nom} · ${t.dateFin}</div></div>${badge(t.statut)}</div>`).join('')||'<div class="empty"><p>Aucune tâche</p></div>'}
    </div></div></div>`;
  mkChart('ch-tc',d.taches_statut,'doughnut');
 }
 else{
  const p=d.ma_performance;
  const curr=d.employe_mois_courant;
  el.innerHTML=`<div class="ptitle">⊞ Mon espace — Agent</div>
   ${curr?`<div class="em-card" style="margin-bottom:18px"><div class="em-month">🏆 Employé du mois — ${MFR[curr.mois]} ${curr.annee}</div><div class="em-name">${curr.nom} ${curr.prenom||''}</div></div>`:''}
   <div class="gstats">
    <div class="stat"><div class="n">${d.nb_taches}</div><div class="l">Tâches assignées</div></div>
    <div class="stat ok"><div class="n">${d.nb_terminees}</div><div class="l">Terminées</div></div>
    <div class="stat warn"><div class="n">${d.nb_notifs}</div><div class="l">Notifs non lues</div></div>
    ${p?`<div class="stat acc"><div class="n">${p.note}/20</div><div class="l">Dernière note</div></div>`:''}
   </div>
   <div class="card"><div class="chead"><h3>Mes tâches en cours</h3></div><div class="cbody">
   ${(d.mes_taches||[]).map(t=>`<div class="${t.priorite==='Haute'?'prio-H':t.priorite==='Urgente'?'prio-U':''}" style="display:flex;justify-content:space-between;align-items:center;padding:11px 8px;border-bottom:1px solid #f0f3f8">
    <div><div style="font-size:.86rem;font-weight:600">${t.libelle}</div><div style="font-size:.74rem;color:var(--mut)">${t.act_nom} · Échéance : <b>${t.dateFin}</b></div></div>
    <div class="acts">${badge(t.statut)}<button class="btn bo bsm" onclick="majStatut(${t.tache_id})">✏️</button><button class="btn bo bsm" onclick="openCR(${t.tache_id})">📄</button></div></div>`).join('')||'<div class="empty"><p>Aucune tâche</p></div>'}
   </div></div>`;
 }
}

// ── BUREAUX ──
async function loadBureaux(){
 const rows=await api('/api/bureaux');
 document.getElementById('pg-bureaux').innerHTML=`
  <div class="ptitle">🏢 Bureaux <button class="btn ba bsm" onclick="addBureau()">+ Ajouter</button></div>
  <div class="card"><table><thead><tr><th>Code</th><th>Nom</th><th>Description</th><th>Personnel</th><th>Activités</th></tr></thead><tbody>
  ${rows.map(r=>`<tr><td><b>${r.code||'—'}</b></td><td><b>${r.nom}</b></td><td style="font-size:.78rem;max-width:220px">${r.description||'—'}</td><td>${r.nb_pers}</td><td>${r.nb_act}</td></tr>`).join('')}
  </tbody></table></div>`;
}
async function addBureau(){
 const n=prompt('Nom du bureau :');if(!n)return;
 const c=prompt('Code (ex: GCS) :');
 await post('/api/bureaux',{nom:n,code:c});loadBureaux();toast('Bureau ajouté','ok');
}

// ── ACTIVITÉS ──
async function loadActivites(){
 const rows=await api('/api/activites');
 const ce=['chef_centre','chef_bureau'].includes(CU.role);
 document.getElementById('pg-activites').innerHTML=`
  <div class="ptitle">📋 Activités ${ce?`<button class="btn ba bsm" onclick="om('mo-activite')">+ Nouvelle</button>`:''}</div>
  <div class="card"><table><thead><tr><th>Nom</th><th>Type</th><th>Bureau</th><th>Début</th><th>Fin</th><th>Statut</th><th>Avancement</th><th>Actions</th></tr></thead><tbody>
  ${rows.map(r=>`<tr><td><b>${r.nom}</b></td><td>${r.type_activite||'—'}</td><td>${r.bureau_nom||'—'}</td>
   <td>${r.dateDebut||'—'}</td><td>${r.dateFin||'—'}</td><td>${badge(r.statut)}</td>
   <td><div style="font-size:.75rem;color:var(--mut)">${r.nb_ok||0}/${r.nb_taches||0} tâches</div>
    <div class="pgbar"><div class="pgfill" style="width:${r.nb_taches?pct(r.nb_ok,r.nb_taches):0}%"></div></div></td>
   <td class="acts">
    <button class="btn bo bsm" onclick="showGantt(${r.activite_id},'${r.nom.replace(/'/g,"\\'")}')">📊 Gantt</button>
    ${ce?`<button class="btn bd" onclick="delAct(${r.activite_id})">🗑</button>`:''}
   </td></tr>`).join('')||'<tr><td colspan="8"><div class="empty"><p>Aucune activité</p></div></td></tr>'}
  </tbody></table></div>`;
}
async function saveActivite(){
 const r=await post('/api/activites',{nom:document.getElementById('ma-n').value,type_activite:document.getElementById('ma-t').value,description:document.getElementById('ma-desc').value,dateDebut:document.getElementById('ma-d').value,dateFin:document.getElementById('ma-f').value,bureau_id:document.getElementById('ma-b').value});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-activite');loadActivites();populateSels();toast('✅ Activité créée','ok');
}
async function delAct(id){if(!confirm('Supprimer ?'))return;await del('/api/activites/'+id);loadActivites();}

async function showGantt(aid,nom){
 const d=await api('/api/activites/'+aid+'/gantt');
 document.getElementById('gantt-t').textContent='📊 Gantt — '+nom;
 if(!d.taches?.length){document.getElementById('gantt-c').innerHTML='<div class="empty"><p>Aucune tâche</p></div>';om('mo-gantt');return;}
 const dates=d.taches.flatMap(t=>[t.dateDebut,t.dateFin]).filter(Boolean).sort();
 const s0=new Date(dates[0]),s1=new Date(dates[dates.length-1]);
 const tot=Math.max(1,(s1-s0)/86400000);
 const sc={Terminee:'#22a06b','En cours':'#e8a020','Non demarre':'#1a3a5c','En retard':'#d93025'};
 let h=`<div style="font-size:.72rem;color:var(--mut);display:flex;justify-content:space-between;margin-bottom:6px"><span>${dates[0]}</span><span>${dates[dates.length-1]}</span></div>`;
 d.taches.forEach(t=>{
  const ts=new Date(t.dateDebut),tf=new Date(t.dateFin);
  const l=Math.max(0,(ts-s0)/86400000/tot*100);
  const w=Math.max(2,(tf-ts)/86400000/tot*100);
  h+=`<div style="display:flex;align-items:center;gap:8px;margin-bottom:7px">
   <div style="min-width:130px;font-size:.76rem;overflow:hidden;text-overflow:ellipsis;white-space:nowrap" title="${t.libelle}">${t.libelle}</div>
   <div style="flex:1;background:#e8edf5;border-radius:4px;height:20px;position:relative">
    <div style="position:absolute;left:${l}%;width:${w}%;height:100%;border-radius:4px;background:${sc[t.statut]||'#888'}" title="${t.statut} | ${t.agents||'Sans agent'}"></div></div>
   ${badge(t.statut)}</div>`;
 });
 document.getElementById('gantt-c').innerHTML=h;om('mo-gantt');
}

// ── TÂCHES ──
async function loadTaches(){
 const rows=await api('/api/taches');
 const cm2=['chef_bureau','controleur'].includes(CU.role);
 const isAg=CU.role==='agent';
 document.getElementById('pg-taches').innerHTML=`
  <div class="ptitle">✅ Tâches ${cm2?`<button class="btn ba bsm" onclick="om('mo-tache')">+ Nouvelle</button>`:''}</div>
  <div class="card"><table><thead><tr><th>Libellé</th><th>Activité</th><th>Début</th><th>Échéance</th><th>Priorité</th><th>Statut</th><th>Agents</th><th>Actions</th></tr></thead><tbody>
  ${rows.map(r=>`<tr class="${r.priorite==='Haute'?'prio-H':r.priorite==='Urgente'?'prio-U':''}"><td><b>${r.libelle}</b></td>
   <td style="font-size:.78rem">${r.act_nom||'—'}</td>
   <td>${r.dateDebut||'—'}</td><td><b>${r.dateFin||'—'}</b></td>
   <td>${badge(r.priorite)}</td><td>${badge(r.statut)}</td>
   <td style="font-size:.76rem;max-width:120px">${r.agents||'—'}</td>
   <td class="acts">
    ${cm2?`<button class="btn bo bsm" onclick="ouvrirGestionAff(${r.tache_id},'${r.libelle.replace(/'/g,"\\'")}')">👥 Affectations</button><button class="btn bd" onclick="delTache(${r.tache_id})">🗑</button>`:''}
    ${isAg?`<button class="btn bo bsm" onclick="majStatut(${r.tache_id})">✏️ Statut</button><button class="btn bo bsm" onclick="openCR(${r.tache_id})">📄 CR</button>`:''}
   </td></tr>`).join('')||'<tr><td colspan="8"><div class="empty"><p>Aucune tâche</p></div></td></tr>'}
  </tbody></table></div>`;
}
async function saveTache(){
 const r=await post('/api/taches',{libelle:document.getElementById('mt-l').value,description:document.getElementById('mt-desc').value,dateDebut:document.getElementById('mt-d').value,dateFin:document.getElementById('mt-f').value,priorite:document.getElementById('mt-p').value,statut:document.getElementById('mt-s').value,activite_id:document.getElementById('mt-a').value});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-tache');loadTaches();toast('✅ Tâche créée','ok');
}
async function delTache(id){if(!confirm('Supprimer ?'))return;await del('/api/taches/'+id);loadTaches();}
async function majStatut(tid){
 const s=prompt('Nouveau statut:\n- Non demarre\n- En cours\n- Terminee\n- En retard');if(!s)return;
 await put('/api/taches/'+tid,{statut:s});loadTaches();toast('Statut mis à jour','ok');
}

// ── AFFECTATIONS ──
async function loadAffecter(){
 const rows=await api('/api/taches');
 document.getElementById('pg-affecter').innerHTML=`
  <div class="ptitle">👤 Gestion des affectations</div>
  <div class="card"><table><thead><tr><th>Tâche</th><th>Activité</th><th>Échéance</th><th>Statut</th><th>Agents affectés</th><th>Actions</th></tr></thead><tbody>
  ${rows.map(r=>`<tr>
   <td><b>${r.libelle}</b></td>
   <td>${r.act_nom||'—'}</td>
   <td>${r.dateFin||'—'}</td>
   <td>${badge(r.statut)}</td>
   <td style="font-size:.76rem">${r.agents||'<em style="color:var(--mut)">Aucun</em>'}</td>
   <td class="acts">
    <button class="btn bp bsm" onclick="ouvrirGestionAff(${r.tache_id},'${r.libelle.replace(/'/g,"\\'")}')">👥 Gérer</button>
    <button class="btn bo bsm" onclick="showHist(${r.tache_id})">📜 Historique</button>
   </td></tr>`).join('')}
  </tbody></table></div>`;
}
async function ouvrirGestionAff(tid, label) {
 document.getElementById('ga-tid').value = tid;
 document.getElementById('ga-titre').textContent = '👥 Gérer — ' + (label||'Tâche #'+tid);
 await rafraichirListeAff(tid);
 om('mo-gerer-aff');
}

async function rafraichirListeAff(tid) {
 const hist = await api('/api/taches/'+tid+'/historique');
 const actifs = hist.filter(h => h.actif == 1);
 const el = document.getElementById('ga-liste');
 if (!actifs.length) {
  el.innerHTML = '<div class="empty"><div class="eico">👤</div><p>Aucun agent affecté actuellement</p></div>';
 } else {
  el.innerHTML = `<table><thead><tr><th>Agent</th><th>Rôle</th><th>Affecté le</th><th>Par</th><th>Action</th></tr></thead><tbody>
  ${actifs.map(a=>`<tr>
   <td><b>${a.agent_nom}</b></td>
   <td><span class="badge bi">${a.role_affect}</span></td>
   <td>${a.date_affectation||'—'}</td>
   <td style="font-size:.76rem">${a.sup_nom||'—'}</td>
   <td><button class="btn bd bsm" onclick="retirerAgent(${tid},${a.personnel_id})">✖ Retirer</button></td>
  </tr>`).join('')}
  </tbody></table>`;
 }
 // Also show "affecter" mini button at bottom
 el.innerHTML += `<div style="margin-top:12px;border-top:1px solid var(--bdr);padding-top:12px">
  <button class="btn ba bsm" onclick="openAff(${tid},'')">+ Affecter un nouvel agent</button>
 </div>`;
}

async function retirerAgent(tid, pid) {
 if (!confirm('Retirer cet agent de la tâche ?')) return;
 const r = await del('/api/taches/'+tid+'/retirer/'+pid);
 if (r.ok) {
  toast('Agent retiré de la tâche','ok');
  await rafraichirListeAff(tid);
  loadAffecter();
 } else {
  toast(r.error||'Erreur','err');
 }
}

async function openAff(tid,label){
 document.getElementById('aff-tid').value=tid;
 document.getElementById('aff-tl').value=label||'Tâche #'+tid;
 // Refresh subordinates list fresh each time
 const pers = await api('/api/personnel');
 const myId = CU.personnel_id;
 const subs = pers.filter(p => Number(p.superieur_id) === Number(myId));
 const opts = subs.length
  ? subs.map(p=>`<option value="${p.personnel_id}">${p.nom} ${p.prenom||''} (${RFR[p.role]||p.role})</option>`).join('')
  : pers.filter(p=>p.personnel_id !== myId).map(p=>`<option value="${p.personnel_id}">${p.nom} ${p.prenom||''} (${RFR[p.role]||p.role})</option>`).join('');
 const sel = document.getElementById('aff-pid');
 if(sel) sel.innerHTML = opts || '<option value="">Aucun subordonné trouvé</option>';
 om('mo-affect');
}
async function saveAffect(){
 const r=await post('/api/taches/'+document.getElementById('aff-tid').value+'/affecter',{personnel_id:document.getElementById('aff-pid').value,role:document.getElementById('aff-r').value});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-affect');loadNotifs();toast('✅ Agent affecté — Notification envoyée','ok');
}
async function showHist(tid){
 const rows=await api('/api/taches/'+tid+'/historique');
 document.getElementById('hist-c').innerHTML=!rows.length?'<div class="empty"><p>Aucune affectation</p></div>':
  `<table><thead><tr><th>Agent</th><th>Rôle</th><th>Affecté le</th><th>Retiré le</th><th>Actif</th><th>Par</th></tr></thead><tbody>
  ${rows.map(r=>`<tr><td>${r.agent_nom}</td><td>${r.role_affect}</td><td>${r.date_affectation||'—'}</td><td>${r.date_retrait||'—'}</td><td>${r.actif?'✅':'❌'}</td><td>${r.sup_nom||'—'}</td></tr>`).join('')}</tbody></table>`;
 om('mo-hist');
}

// ── PERSONNEL ──
async function loadPersonnel(){
 const rows=await api('/api/personnel');
 const ce=['chef_centre','chef_bureau'].includes(CU.role);
 document.getElementById('pg-personnel').innerHTML=`
  <div class="ptitle">👥 Personnel ${CU.role==='chef_centre'?`<button class="btn ba bsm" onclick="om('mo-personnel')">+ Ajouter</button>`:''}</div>
  <div class="card"><table><thead><tr><th>Nom</th><th>Rôle</th><th>Fonction</th><th>Bureau</th><th>Email</th><th>Tél.</th><th>Depuis</th><th>Actions</th></tr></thead><tbody>
  ${rows.map(r=>`<tr><td><div style="display:flex;align-items:center;gap:8px">${r.photo_profil?`<img src="${r.photo_profil}" style="width:32px;height:32px;border-radius:50%;object-fit:cover">`:av(r.nom,r.prenom,r.role)}<b>${r.nom} ${r.prenom||''}</b></div></td>
   <td><span class="role-chip rc-${r.role}">${RFR[r.role]||r.role}</span></td>
   <td>${r.fonction||'—'}</td><td>${r.bureau_nom||'—'}</td><td style="font-size:.76rem">${r.email}</td><td>${r.telephone||'—'}</td><td>${r.annee_integration||'—'}</td>
   <td class="acts">
    <button class="btn bsuc bsm" onclick="openNoter(${r.personnel_id},'${(r.nom+' '+(r.prenom||'')).trim().replace(/'/g,"\\'")}')">⭐ Éval.</button>
    <button class="btn ba bsm" onclick="openEM(${r.personnel_id},'${(r.nom+' '+(r.prenom||'')).trim().replace(/'/g,"\\'")}')">🏆</button>
    ${CU.role==='chef_centre'?`<button class="btn bd" onclick="delPers(${r.personnel_id})">🗑 Retirer</button>`:''}
   </td></tr>`).join('')}
  </tbody></table></div>`;
}
function previewPhoto(input, imgId, iconId, b64Id) {
 const file = input.files[0]; if (!file) return;
 const reader = new FileReader();
 reader.onload = e => {
  document.getElementById(imgId).src = e.target.result;
  document.getElementById(imgId).style.display = 'block';
  document.getElementById(iconId).style.display = 'none';
  document.getElementById(b64Id).value = e.target.result;
 };
 reader.readAsDataURL(file);
}

async function savePersonnel(){
 const r=await post('/api/personnel',{
  nom:document.getElementById('mp-n').value,
  prenom:document.getElementById('mp-pr').value,
  telephone:document.getElementById('mp-t').value,
  dateNaissance:document.getElementById('mp-dob').value,
  fonction:document.getElementById('mp-f').value,
  role:document.getElementById('mp-r').value,
  bureau_id:document.getElementById('mp-b').value,
  superieur_id:document.getElementById('mp-s').value||null
 });
 if(r.error){toast(r.error,'err');return;}
 // Upload photo if provided
 const photo = document.getElementById('mp-photo-b64').value;
 if(photo && r.personnel_id) await post('/api/personnel/'+r.personnel_id+'/photo', {photo});
 cm('mo-personnel');loadPersonnel();populateSels();
 toast(`✅ Personnel ajouté — Email: ${r.email} | MDP: ${r.password}`,'ok');
 // Reset photo
 document.getElementById('mp-photo-b64').value='';
 document.getElementById('mp-photo-img').style.display='none';
 document.getElementById('mp-photo-icon').style.display='';
}
async function delPers(id){if(!confirm('Supprimer ?'))return;await del('/api/personnel/'+id);loadPersonnel();}

// ── PERFORMANCES ──
async function loadPerformances(){
 const rows=await api('/api/performances');
 const ce=['chef_bureau','controleur'].includes(CU.role);
 document.getElementById('pg-performances').innerHTML=`
  <div class="ptitle">📊 Performances ${ce?`<button class="btn ba bsm" onclick="openNoterModal()">+ Évaluer</button>`:''}</div>
  <div class="card"><table><thead><tr><th>Agent</th><th>Note</th><th>Efficacité</th><th>Prime</th><th>Commentaire</th><th>Mois</th>${CU.role!=='agent'?'<th>Évaluateur</th>':''}</tr></thead><tbody>
  ${rows.map(r=>`<tr><td><b>${r.nom||'Moi'} ${r.prenom||''}</b></td>
   <td><b style="font-size:1rem;color:${r.note>=15?'#22a06b':r.note>=10?'#e07b28':'#d93025'}">${r.note}</b>/20</td>
   <td><div style="display:flex;align-items:center;gap:7px"><div class="pgbar" style="width:70px"><div class="pgfill" style="width:${r.efficacite||0}%;background:${r.efficacite>=75?'#22a06b':r.efficacite>=50?'#e07b28':'#d93025'}"></div></div>${r.efficacite||0}%</div></td>
   <td>${r.prime||'—'}</td><td style="max-width:200px;font-size:.78rem">${r.commentaire||'—'}</td>
   <td>${r.mois?MFR[r.mois]+' '+r.annee:r.date_eval||'—'}</td>
   ${CU.role!=='agent'?`<td>${r.eval_nom||'—'}</td>`:''}</tr>`).join('')||
   '<tr><td colspan="7"><div class="empty"><p>Aucune évaluation</p></div></td></tr>'}
  </tbody></table></div>`;
}
function openNoterModal(){document.getElementById('n-pid').value='';document.getElementById('n-nom').value='';om('mo-noter');}
function openNoter(pid,nom){document.getElementById('n-pid').value=pid;document.getElementById('n-nom').value=nom;om('mo-noter');}
async function saveNote(){
 const pid=document.getElementById('n-pid').value;if(!pid){toast('Sélectionnez un agent','err');return;}
 const r=await post('/api/performances',{personnel_id:pid,efficacite:document.getElementById('n-eff').value,note:document.getElementById('n-note').value,prime:document.getElementById('n-prime').value,commentaire:document.getElementById('n-comm').value});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-noter');loadPerformances();loadNotifs();toast('✅ Évaluation enregistrée + notifié','ok');
}

// ── EMPLOYÉ DU MOIS ──
async function loadEmployeMois(){
 const rows=await api('/api/employe-mois');
 const ce=['chef_centre','chef_bureau'].includes(CU.role);
 document.getElementById('pg-employe-mois').innerHTML=`
  <div class="ptitle">🏆 Employé du mois ${ce?`<button class="btn ba bsm" onclick="om('mo-em')">+ Désigner</button>`:''}</div>
  <div style="display:grid;grid-template-columns:repeat(auto-fill,minmax(260px,1fr));gap:14px;margin-bottom:20px">
  ${rows.map(em=>`<div class="em-card">
   <div class="em-month">${MFR[em.mois]} ${em.annee}</div>
   <div class="em-name">${em.nom} ${em.prenom||''}</div>
   <div class="em-bureau">${em.bureau_nom} · ${em.fonction||''}</div>
   <div class="em-note" style="margin-top:8px;font-size:.78rem"><b>Note : ${em.note_finale}/20</b></div>
   <div class="em-note" style="margin-top:4px;font-size:.76rem;opacity:.85">${em.motif||''}</div>
   <div class="em-note" style="margin-top:6px;font-size:.7rem;opacity:.7">Désigné par : ${em.designateur_nom||'—'}</div>
  </div>`).join('')||'<p style="color:var(--mut);grid-column:1/-1">Aucune désignation pour l\'instant.</p>'}
  </div>`;
}
function openEM(pid,nom){
 document.getElementById('em-pid').value=pid;
 document.getElementById('em-motif').value='';
 om('mo-em');
}
async function saveEM(){
 const r=await post('/api/employe-mois',{personnel_id:document.getElementById('em-pid').value,mois:document.getElementById('em-mois').value,annee:document.getElementById('em-annee').value,note_finale:document.getElementById('em-note').value,motif:document.getElementById('em-motif').value});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-em');loadEmployeMois();loadNotifs();toast('🏆 Employé du mois désigné ! Notification envoyée.','ok');
}

// ── TÉLÉPAIEMENT ──
async function loadTelepaiement(){
 const [rows,stats]=await Promise.all([api('/api/telepaiements'),api('/api/telepaiements/stats')]);
 const total=fmt(Math.round((stats.total_attendu||0)*1000000));
 const paye=fmt(Math.round((stats.total_recouvre||0)*1000000));
 const tauxRec=stats.total_attendu?Math.round(stats.total_recouvre/stats.total_attendu*100):0;
 const ce=['chef_centre','chef_bureau','controleur','agent'].includes(CU.role);
 document.getElementById('pg-telepaiement').innerHTML=`
  <div class="ptitle">💳 Télépaiements <button class="btn ba bsm" onclick="om('mo-tp')">+ Nouveau</button></div>
  <div class="gstats" style="margin-bottom:16px">
   <div class="stat"><div class="n">${rows.length}</div><div class="l">Dossiers</div></div>
   <div class="stat ok"><div class="n">${total} F</div><div class="l">Montant attendu</div></div>
   <div class="stat acc"><div class="n">${paye} F</div><div class="l">Montant recouvré</div></div>
   <div class="stat warn"><div class="n">${tauxRec}%</div><div class="l">Taux recouvrement</div></div>
  </div>
  <div class="grids2" style="margin-bottom:16px">
   <div class="card"><div class="chead"><h3>Répartition par statut</h3></div><div class="cbody"><div class="cht"><canvas id="ch-tp-s"></canvas></div></div></div>
   <div class="card"><div class="chead"><h3>Par type d'impôt (Millions FCFA)</h3></div><div class="cbody"><div class="cht"><canvas id="ch-tp-t"></canvas></div></div></div>
  </div>
  <div class="card"><table><thead><tr><th>Référence</th><th>Contribuable</th><th>NIF</th><th>Type impôt</th><th>Montant dû</th><th>Reçu</th><th>Statut</th><th>Échéance</th><th>Bureau</th><th>Actions</th></tr></thead><tbody>
  ${rows.map(r=>`<tr>
   <td><b>${r.reference}</b></td>
   <td>${r.contribuable_nom}</td>
   <td style="font-size:.76rem">${r.contribuable_nif||'—'}</td>
   <td style="font-size:.78rem">${r.type_impot}</td>
   <td><b>${fmt(r.montant)}</b></td>
   <td class="tp-statut-${r.statut==='Paye'?'paye':r.statut==='En retard'?'retard':'partiel'}">${fmt(r.montant_paye)}</td>
   <td>${badge(r.statut)}</td>
   <td>${r.date_echeance||'—'}</td>
   <td style="font-size:.76rem">${r.bureau_nom||'—'}</td>
   <td><button class="btn bp bsm" onclick="openTPUpdate(${r.tp_id},${r.montant},'${r.statut}')">✏️ MàJ</button></td>
  </tr>`).join('')||'<tr><td colspan="10"><div class="empty"><p>Aucun dossier</p></div></td></tr>'}
  </tbody></table></div>`;
 if(stats.par_statut?.length)mkChart('ch-tp-s',stats.par_statut,'doughnut');
 if(stats.par_type?.length)mkChart('ch-tp-t',stats.par_type,'bar');
}
async function saveTP(){
 const r=await post('/api/telepaiements',{contribuable_nom:document.getElementById('tp-cn').value,contribuable_nif:document.getElementById('tp-nif').value,type_impot:document.getElementById('tp-type').value,montant:document.getElementById('tp-mt').value,bureau_id:document.getElementById('tp-b').value,date_echeance:document.getElementById('tp-ech').value,notes:document.getElementById('tp-notes').value});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-tp');loadTelepaiement();toast(`✅ Télépaiement créé — Réf: ${r.reference}`,'ok');
}
function openTPUpdate(id,montant,statut){
 document.getElementById('tpu-id').value=id;
 document.getElementById('tpu-mt').value=montant;
 document.getElementById('tpu-statut').value=statut==='En attente'?'Partiel':statut;
 om('mo-tp-update');
}
async function saveTPUpdate(){
 const id=document.getElementById('tpu-id').value;
 const mt=parseFloat(document.getElementById('tpu-mt').value);
 const r=await put('/api/telepaiements/'+id,{statut:document.getElementById('tpu-statut').value,montant_paye:mt,mode_paiement:document.getElementById('tpu-mode').value,notes:document.getElementById('tpu-notes').value});
 cm('mo-tp-update');loadTelepaiement();toast('✅ Paiement mis à jour','ok');
}

// ── SIGNALEMENTS ──
async function loadSignalements(){
 const rows=await api('/api/signalements');
 const isN=['chef_centre','chef_bureau','controleur'].includes(CU.role);
 document.getElementById('pg-signalements').innerHTML=`
  <div class="ptitle">⚠️ Signalements <button class="btn ba bsm" onclick="om('mo-signal')">+ Signaler</button></div>
  <div class="card"><table><thead><tr><th>Description</th>${isN?'<th>Auteur</th>':''}<th>Tâche</th><th>Date</th><th>Statut</th>${isN?'<th>Actions</th>':''}</tr></thead><tbody>
  ${rows.map(r=>`<tr><td style="max-width:220px">${r.description}</td>${isN?`<td>${r.auteur||'—'}</td>`:''}
   <td>${r.tache_lib||'—'}</td><td>${r.dateEnvoie||'—'}</td><td>${badge(r.statut)}</td>
   ${isN&&r.statut==='Ouvert'?`<td><button class="btn bsuc bsm" onclick="traiterSignal(${r.signalement_id})">✅ Traiter</button></td>`:(isN?'<td>—</td>':'')}
  </tr>`).join('')||'<tr><td colspan="6"><div class="empty"><p>Aucun signalement</p></div></td></tr>'}
  </tbody></table></div>`;
}
async function saveSignal(){
 const r=await post('/api/signalements',{description:document.getElementById('sg-d').value,tache_id:document.getElementById('sg-t').value||null});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-signal');loadSignalements();toast('✅ Signalement envoyé au N+1','ok');
}
async function traiterSignal(id){
 const rep=prompt('Réponse / action entreprise :');
 await put('/api/signalements/'+id+'/repondre',{statut:'Traite',reponse:rep});loadSignalements();toast('Signalement traité','ok');
}

// ── PROPOSITIONS ──
async function loadPropositions(){
 const rows=await api('/api/propositions');
 const isN=['chef_centre','chef_bureau'].includes(CU.role);
 document.getElementById('pg-propositions').innerHTML=`
  <div class="ptitle">💡 ${isN?'Propositions reçues':'Mes propositions'} ${!isN?`<button class="btn ba bsm" onclick="om('mo-prop')">+ Proposer</button>`:''}</div>
  <div class="card"><table><thead><tr><th>Description</th><th>Type</th>${isN?'<th>Proposé par</th>':'<th>Destinataire</th>'}<th>Date</th><th>Statut</th>${isN?'<th>Actions</th>':''}</tr></thead><tbody>
  ${rows.map(r=>`<tr><td style="max-width:250px">${r.description}</td><td>${r.type_activite}</td>
   <td>${r.proposeur?(r.proposeur+' '+(r.proposeur_prenom||'')):r.destinataire||'—'}</td>
   <td>${r.date_prop||'—'}</td><td>${badge(r.statut)}</td>
   ${isN&&r.statut==='En attente'?`<td class="acts"><button class="btn bsuc bsm" onclick="repProp(${r.prop_id},'Acceptee')">✅</button><button class="btn bd" onclick="repProp(${r.prop_id},'Refusee')">✗</button></td>`:(isN?'<td>—</td>':'')}
  </tr>`).join('')||'<tr><td colspan="6"><div class="empty"><p>Aucune proposition</p></div></td></tr>'}
  </tbody></table></div>`;
}
async function saveProp(){
 const r=await post('/api/propositions',{type_activite:document.getElementById('pp-t').value,description:document.getElementById('pp-d').value});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-prop');loadPropositions();toast('✅ Proposition envoyée au N+1','ok');
}
async function repProp(id,statut){
 const c=prompt(statut==='Acceptee'?'Message d\'acceptation (optionnel) :':'Motif du refus :');
 await put('/api/propositions/'+id+'/repondre',{statut,commentaire:c});loadPropositions();loadNotifs();toast(`Proposition ${statut.toLowerCase()}`,'ok');
}

// ── COMPTES-RENDUS ──
function openCR(tid){document.getElementById('cr-t').value=tid||'';om('mo-cr');}
async function loadCR(){
 const rows=await api('/api/comptes-rendus');
 const isReceiver = ['chef_bureau','controleur'].includes(CU.role);
 const canSubmit = CU.role !== 'chef_centre';
 document.getElementById('pg-cr').innerHTML=`
  <div class="ptitle">📄 Comptes-rendus
   ${canSubmit?`<button class="btn ba bsm" onclick="om('mo-cr')">+ Nouveau</button>`:''}
   ${rows.length?`<span class="badge bi" style="margin-left:8px">${rows.length} au total</span>`:''}
  </div>
  <div class="card"><table><thead><tr>
   <th>Auteur</th>
   ${isReceiver||CU.role==='chef_centre'?'<th>Bureau</th>':''}
   <th>Tâche</th><th>Date</th><th>Statut</th><th>Contenu</th>
  </tr></thead><tbody>
  ${rows.map(r=>`<tr>
   <td><b>${r.nom?(r.nom+' '+(r.prenom||'')):'Moi'}</b></td>
   ${isReceiver||CU.role==='chef_centre'?`<td style="font-size:.76rem">${r.bureau_nom||'—'}</td>`:''}
   <td>${r.tache_lib||'—'}</td>
   <td>${r.dateRendue||'—'}</td>
   <td>${badge(r.statut)}</td>
   <td style="max-width:260px;font-size:.78rem">${(r.contenu||'').substring(0,120)}${(r.contenu||'').length>120?'…':''}</td>
  </tr>`).join('')||'<tr><td colspan="6"><div class="empty"><div class="eico">📄</div><p>Aucun compte-rendu</p></div></td></tr>'}
  </tbody></table></div>`;
}
async function saveCR(){
 const r=await post('/api/comptes-rendus',{contenu:document.getElementById('cr-c').value,tache_id:document.getElementById('cr-t').value||null});
 if(r.error){toast(r.error,'err');return;}
 cm('mo-cr');loadCR();loadNotifs();toast('✅ Compte-rendu soumis — N+1 notifié','ok');
}

// ── AVIS ──
async function loadAvis(){
 const rows=await api('/api/avis');
 document.getElementById('pg-avis').innerHTML=`
  <div class="ptitle">⭐ Avis sur l'application <button class="btn ba bsm" onclick="om('mo-avis')">+ Donner un avis</button></div>
  <div class="card"><table><thead><tr><th>Agent</th><th>Bureau</th><th>Note</th><th>Commentaire</th><th>Date</th></tr></thead><tbody>
  ${rows.map(r=>`<tr><td>${r.nom?(r.nom+' '+(r.prenom||'')):'—'}</td><td>${r.bureau_nom||'—'}</td><td>${stars(r.note)}</td><td>${r.commentaire||'—'}</td><td>${r.dateEnvoie||'—'}</td></tr>`).join('')||
  '<tr><td colspan="5"><div class="empty"><p>Aucun avis</p></div></td></tr>'}
  </tbody></table></div>`;
}
async function saveAvis(){
 await post('/api/avis',{commentaire:document.getElementById('av-c').value,note:document.getElementById('av-n').value});
 cm('mo-avis');loadAvis();toast('✅ Avis soumis. Merci !','ok');
}

// ── POPULATE SELECTS ──
async function populateSels(){
 const [bur,pers,act,taches]=await Promise.all([api('/api/bureaux'),api('/api/personnel'),api('/api/activites'),api('/api/taches')]);
 const bO=bur.map(b=>`<option value="${b.bureau_id}">${b.nom}</option>`).join('');
 const pO=pers.map(p=>`<option value="${p.personnel_id}">${p.nom} ${p.prenom||''} (${RFR[p.role]||p.role})</option>`).join('');
 const aO=act.map(a=>`<option value="${a.activite_id}">${a.nom}</option>`).join('');
 const tO=taches.map(t=>`<option value="${t.tache_id}">${t.libelle}</option>`).join('');
 // Each N+1 can only affect their direct subordinates
 // chef_centre sees all chefs_bureau (those who have chef_centre as superieur)
 // chef_bureau sees their controleurs, controleur sees their agents
 const myId = CU.personnel_id;
 const subP = pers.filter(p => Number(p.superieur_id) === Number(myId));
 const subO = subP.length
  ? subP.map(p=>`<option value="${p.personnel_id}">${p.nom} ${p.prenom||''} (${RFR[p.role]||p.role})</option>`).join('')
  : pers.map(p=>`<option value="${p.personnel_id}">${p.nom} ${p.prenom||''} (${RFR[p.role]||p.role})</option>`).join('');
 ['ma-b','tp-b','mp-b'].forEach(id=>{const el=document.getElementById(id);if(el)el.innerHTML=bO;});
 ['mt-a'].forEach(id=>{const el=document.getElementById(id);if(el)el.innerHTML=aO;});
 const mpS=document.getElementById('mp-s');if(mpS)mpS.innerHTML='<option value="">-- Aucun --</option>'+pO;
 const aP=document.getElementById('aff-pid');if(aP)aP.innerHTML=subO||pO;
 const emP=document.getElementById('em-pid');if(emP)emP.innerHTML=pO;
 const sgT=document.getElementById('sg-t');if(sgT)sgT.innerHTML='<option value="">-- Aucune --</option>'+tO;
 const crT=document.getElementById('cr-t');if(crT)crT.innerHTML='<option value="">-- Général --</option>'+tO;
}

async function loadProfil() {
 const p = await api('/api/personnel/'+CU.personnel_id);
 if(!p || p.error) return;
 const el = document.getElementById('pg-profil');
 const photoHtml = p.photo_profil
  ? `<img src="${p.photo_profil}" style="width:110px;height:110px;border-radius:50%;object-fit:cover;border:3px solid var(--acc)">`
  : `<div style="width:110px;height:110px;border-radius:50%;background:var(--p);color:#C9A227;display:flex;align-items:center;justify-content:center;font-size:2.5rem;font-weight:900;border:3px solid var(--acc)">${((p.prenom||'?')[0]+(p.nom||'?')[0]).toUpperCase()}</div>`;
 el.innerHTML = `
  <div class="ptitle">🪪 Informations personnelles</div>
  <div class="card" style="max-width:600px">
   <div class="chead"><h3>Mon profil</h3></div>
   <div class="cbody">
    <div style="display:flex;gap:24px;align-items:center;margin-bottom:20px;flex-wrap:wrap">
     <div style="text-align:center">
      <div id="profil-photo-wrap" style="margin-bottom:8px">${photoHtml}</div>
      <input type="file" id="profil-photo-input" accept="image/*" style="display:none" onchange="uploadMyPhoto(this)">
      <button class="btn bo bsm" onclick="document.getElementById('profil-photo-input').click()">📷 Changer la photo</button>
     </div>
     <div style="flex:1;min-width:200px">
      <div style="font-size:1.4rem;font-weight:800;color:var(--p)">${p.prenom||''} ${p.nom}</div>
      <div style="margin:4px 0"><span class="role-chip rc-${p.role}">${RFR[p.role]||p.role}</span></div>
      <div style="font-size:.85rem;color:var(--mut);margin-top:6px">${p.fonction||'—'}</div>
      <div style="font-size:.82rem;margin-top:4px">📧 ${p.email}</div>
      ${p.telephone?`<div style="font-size:.82rem">📱 ${p.telephone}</div>`:''}
     </div>
    </div>
    <hr style="border:none;border-top:1px solid var(--bdr);margin:12px 0">
    <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px">
     <div class="fg"><label>Prénom</label><input id="prf-prenom" value="${p.prenom||''}"></div>
     <div class="fg"><label>Nom</label><input id="prf-nom" value="${p.nom||''}"></div>
     <div class="fg"><label>Téléphone</label><input id="prf-tel" value="${p.telephone||''}"></div>
     <div class="fg"><label>Date de naissance</label><input type="date" id="prf-dob" value="${p.dateNaissance||''}"></div>
     <div class="fg" style="grid-column:1/-1"><label>Adresse</label><input id="prf-adr" value="${p.adresse||''}"></div>
    </div>
    <div style="margin-top:16px;display:flex;gap:10px;justify-content:flex-end">
     <button class="btn bp" onclick="saveProfil()">💾 Enregistrer</button>
     <button class="btn bo" onclick="changePassword()">🔒 Changer mot de passe</button>
    </div>
   </div>
  </div>`;
}

async function uploadMyPhoto(input) {
 const file = input.files[0]; if(!file) return;
 const reader = new FileReader();
 reader.onload = async e => {
  const r = await post('/api/personnel/'+CU.personnel_id+'/photo', {photo: e.target.result});
  if(r.ok) { toast('Photo mise à jour','ok'); loadProfil(); }
  else toast('Erreur upload photo','err');
 };
 reader.readAsDataURL(file);
}

async function saveProfil() {
 const payload = {
  prenom: document.getElementById('prf-prenom').value,
  nom: document.getElementById('prf-nom').value,
  telephone: document.getElementById('prf-tel').value,
  adresse: document.getElementById('prf-adr').value,
  dateNaissance: document.getElementById('prf-dob').value
 };
 const r = await put('/api/personnel/'+CU.personnel_id, payload);
 if(r && r.ok) {
  toast('✅ Profil mis à jour','ok');
  CU.nom = payload.nom;
  CU.prenom = payload.prenom;
  document.getElementById('uname').textContent = `${CU.prenom||''} ${CU.nom}`;
 } else {
  toast(r && r.error ? r.error : 'Erreur lors de la mise à jour','err');
 }
}

async function changePassword() {
 const old = prompt('Mot de passe actuel :'); if(!old) return;
 const nw = prompt('Nouveau mot de passe :'); if(!nw) return;
 const r = await post('/api/change-password', {old_password: old, new_password: nw});
 if(r.ok) toast('Mot de passe changé avec succès','ok');
 else toast(r.error||'Erreur','err');
}

const LOADS={dashboard:loadDashboard,bureaux:loadBureaux,activites:loadActivites,taches:loadTaches,affecter:loadAffecter,personnel:loadPersonnel,performances:loadPerformances,'employe-mois':loadEmployeMois,telepaiement:loadTelepaiement,signalements:loadSignalements,propositions:loadPropositions,cr:loadCR,avis:loadAvis,profil:loadProfil};

document.getElementById('l-p').addEventListener('keydown',e=>{if(e.key==='Enter')doLogin();});
</script></body></html>
"""

@app.route('/')
def index():
    return render_template_string(HTML)

print('✅ Application Flask prête')
print(f'   {len(list(app.url_map.iter_rules()))} routes définies')

✅ Application Flask prête
   33 routes définies


## 🌐 Cellule 4 — Démarrage & URL publique

In [6]:
from pyngrok import ngrok
import threading, time

# ─────────────────────────────────────────────────────────
# Collez votre token ici : https://dashboard.ngrok.com
NGROK_TOKEN = '3D5m5VBF09wS7J8pZ9Zgtru1D7K_4BGJkk53iEWjruhdvnZFh'
# ─────────────────────────────────────────────────────────

ngrok.kill()
if NGROK_TOKEN and NGROK_TOKEN != 'VOTRE_TOKEN_NGROK_ICI':
    ngrok.set_auth_token(NGROK_TOKEN)

threading.Thread(target=lambda: app.run(port=5000, use_reloader=False, debug=False), daemon=True).start()
time.sleep(2)

try:
    url = ngrok.connect(5000)
    print('═'*60)
    print('🌍  TERRAFISC est en ligne :')
    print(f'    👉  {url}')
    print('═'*60)
except Exception as e:
    print(f'⚠️  ngrok: {e}')
    print('   → Colab : Runtime → Ports → Port 5000')

print('')
print('📋  COMPTES  (mot de passe : admin123)')
print('  ┌────────────────────────────────────────────────────┐')
print('  │  chef.centre@terrafisc.sn   Chef de Centre        │')
print('  │  cb.gcs@terrafisc.sn        Chef Bureau GCS       │')
print('  │  cb.rec@terrafisc.sn        Chef Bureau Recouvr.  │')
print('  │  cb.dom@terrafisc.sn        Chef Bureau Domaines  │')
print('  │  cb.cof@terrafisc.sn        Chef Conserv. Fonc.   │')
print('  │  cb.cad@terrafisc.sn        Chef Cadastre         │')
print('  │  ctrl.gcs1@terrafisc.sn     Contrôleur GCS        │')
print('  │  ctrl.rec1@terrafisc.sn     Contrôleur REC        │')
print('  │  agent.gcs1@terrafisc.sn    Agent GCS             │')
print('  │  agent.rec1@terrafisc.sn    Agent Recouvrement    │')
print('  └────────────────────────────────────────────────────┘')

 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.


════════════════════════════════════════════════════════════
🌍  TERRAFISC est en ligne :
    👉  NgrokTunnel: "https://sloped-snagged-handwash.ngrok-free.dev" -> "http://localhost:5000"
════════════════════════════════════════════════════════════

📋  COMPTES  (mot de passe : admin123)
  ┌────────────────────────────────────────────────────┐
  │  chef.centre@terrafisc.sn   Chef de Centre        │
  │  cb.gcs@terrafisc.sn        Chef Bureau GCS       │
  │  cb.rec@terrafisc.sn        Chef Bureau Recouvr.  │
  │  cb.dom@terrafisc.sn        Chef Bureau Domaines  │
  │  cb.cof@terrafisc.sn        Chef Conserv. Fonc.   │
  │  cb.cad@terrafisc.sn        Chef Cadastre         │
  │  ctrl.gcs1@terrafisc.sn     Contrôleur GCS        │
  │  ctrl.rec1@terrafisc.sn     Contrôleur REC        │
  │  agent.gcs1@terrafisc.sn    Agent GCS             │
  │  agent.rec1@terrafisc.sn    Agent Recouvrement    │
  └────────────────────────────────────────────────────┘


---
## 📖 Récapitulatif des fonctionnalités

### 🏢 5 Bureaux réels
| Code | Bureau |
|---|---|
| GCS | Gestion des Contribuables et Services |
| REC | Recouvrement |
| DOM | Bureau des Domaines |
| COF | Conservation Foncière |
| CAD | Cadastre |

### 🏆 Employé du mois
- Chef de Centre et Chefs de Bureau peuvent désigner l'employé du mois
- L'agent désigné reçoit une **notification sonore/vibration** automatique
- Historique visible par tous sur la page dédiée
- Affiché en bannière sur le tableau de bord

### 💳 Télépaiement
- Gestion des dossiers de paiement fiscal (IS, TVA, taxe foncière, etc.)
- Suivi des statuts : En attente / Partiel / Payé / En retard
- Modes de paiement : Virement, Mobile Money (Wave/Orange), Carte, Espèces
- Statistiques visuelles par statut, type d'impôt et bureau
- Tableau de bord avec taux de recouvrement

### 🔔 Notifications sonores + vibration
- Déclenchées à chaque affectation, évaluation, désignation employé du mois
- Son adapté au type (succès = aigu, erreur = grave)
- Vibration sur mobile